Reading data


In [0]:
%python
facilities = spark.read.table("databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities")

india_post_pincode_directory = spark.read.table(
    "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.india_post_pincode_directory"
)

nfhs_5_district_health_indicators = spark.read.table(
    "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.nfhs_5_district_health_indicators"
)

Display facilities

In [0]:
%python
display(facilities)

unique_id source_types source_ids source_content_id name organization_type content_table_id phone_numbers officialPhone email websites officialWebsite yearEstablished acceptsVolunteers facebookLink address_line1 address_line2 address_line3 address_city address_stateOrRegion address_zipOrPostcode address_country address_countryCode countries facilityTypeId operatorTypeId affiliationTypeIds description area numberDoctors capacity specialties procedure equipment capability recency_of_page_update distinct_social_media_presence_count affiliated_staff_presence custom_logo_presence number_of_facts_about_the_organization post_metrics_most_recent_social_media_post_date post_metrics_post_count engagement_metrics_n_followers engagement_metrics_n_likes engagement_metrics_n_engagements source coordinates latitude longitude cluster_id source_urls b8a5401f-42f1-422a-8cd9-686a15b4cb76 ["dynamic","overture","constant","mongo_facility","dynamic","overture","constant","mongo_facility","dynamic","overture","constant","mongo_facility","dynamic","overture","constant","mongo_facility","overture","overture","overture","overture","overture","overture","overture","overture","overture","overture","overture","overture","overture","overture","overture","dynamic","dynamic","dynamic","dynamic","dynamic","overture","dynamic","overture","dynamic","overture","dynamic","overture","dynamic","dynamic","dynamic","dynamic","dynamic","dynamic","dynamic"] ["08f42d99986e2b6d03862b16be6f4556","08f3cb3a2a1a139503b31af626a9e452","08f52b1ca500bb5c034e6665048ce729","605030acb13d98c445519412","08f6088501329d6203f8351114f22d05","08f3d8dcc049d12d03e61e3db26d807b","08f3f5bb4b61d175032dd20192fbba30","08f42d99986e2b6d03862b16be6f4556","08f3cb3a2a1a139503b31af626a9e452","08f52b1ca500bb5c034e6665048ce729","605030acb13d98c445519412","08f6088501329d6203f8351114f22d05","08f3d8dcc049d12d03e61e3db26d807b","08f3f5bb4b61d175032dd20192fbba30","08f42d99986e2b6d03862b16be6f4556","08f3cb3a2a1a139503b31af626a9e452","08f52b1ca500bb5c034e6665048ce729","605030acb13d98c445519412","08f6088501329d6203f8351114f22d05","08f3d8dcc049d12d03e61e3db26d807b","08f3f5bb4b61d175032dd20192fbba30","08f42d99986e2b6d03862b16be6f4556","08f3cb3a2a1a139503b31af626a9e452","08f52b1ca500bb5c034e6665048ce729","605030acb13d98c445519412","08f6088501329d6203f8351114f22d05","08f3d8dcc049d12d03e61e3db26d807b","08f3f5bb4b61d175032dd20192fbba30","08f42d98adbb562203277b401e869481","08f42d98adbb562203277b401e869481","08f42d98adbb562203277b401e869481","08f42d98adbb562203277b401e869481","08f3c5e6d9862b1103728d291ec91cb4","08f3c5e6d9862b1103728d291ec91cb4","08f3c5e6d9862b1103728d291ec91cb4","08f3c5e6d9862b1103728d291ec91cb4","08f3c5e6d9862b1103728d291ec91cb4","08f3c5e6d9862b1103728d291ec91cb4","08f3c5e6d9862b1103728d291ec91cb4","08f42d98adbb562203277b401e869481","08f42d98adbb562203277b401e869481","08f42d98adbb562203277b401e869481","08f42d98adbb562203277b401e869481","08f3d8ba04c91b1803b606d82aa78179","08f452207482cc58038440503a9c606a","08f3d8ba04c91b1803b606d82aa78179","08f452207482cc58038440503a9c606a","08f3d8ba04c91b1803b606d82aa78179","08f452207482cc58038440503a9c606a","08f3d8ba04c91b1803b606d82aa78179"] c2ada12b-11f7-45f2-93b6-94f7e8853c7a Aravind Eye Hospital facility c2ada12b-11f7-45f2-93b6-94f7e8853c7a ["+911123233400","+911123232400","+911126142832","+911126141119","+911120895995","+919910631876","+911123236000","+911123236000","+911122890612","+911122890611","+911122890789","+911122890612","+911122890611","+911122890789","+911122588383","+911123236000","+911124334225","+911120838218","+911122890612","+911122890611","+911122890789","+911122890612","+911122890611","+911122890789","+911122586262","+911122588383","+919910631876","+911120895995","+911123233925","+911123217081","+911123236000","+911123236000","+911123236000","+911126730453","+911126730160","+918422801801","+919982782555","+918104983787","+914068106587","+914068106585","+914068106589","+914061656565","+911125735205","+911125861463","+911126925858","+911126588500",

In [0]:
%sql
SELECT COUNT(*)
FROM databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities

COUNT(*)
10088


Read & Confirm Pin

In [0]:
%python
from pyspark.sql import functions as F

source_schema = (
    "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset"
)

facilities = spark.read.table(f"{source_schema}.facilities")
postal_directory = spark.read.table(f"{source_schema}.india_post_pincode_directory")


def clean_text(column):
    value = F.trim(column.cast("string"))
    return F.when((value != "") & (F.lower(value) != "null"), value)


def normalize_place(column):
    return F.regexp_replace(F.lower(clean_text(column)), r"[^\p{L}\p{N}]", "")


street = F.concat_ws(
    ", ",
    *[clean_text(F.col(f"address_line{i}")) for i in (1, 2, 3)],
)

clean_facilities = (
    facilities
    .withColumn("facility_id", clean_text(F.col("unique_id")))
    .withColumn("facility_name", clean_text(F.col("name")))
    .withColumn("street_address", F.when(street != "", street))
    .withColumn("city", clean_text(F.col("address_city")))
    .withColumn("reported_region", clean_text(F.col("address_stateOrRegion")))
    .withColumn("pin_code", clean_text(F.col("address_zipOrPostcode")))
    .withColumn("official_phone", clean_text(F.col("officialPhone")))
    .withColumn("reported_doctor_count", F.expr("try_cast(numberDoctors AS BIGINT)"))
    .withColumn("latitude_clean", F.expr("try_cast(latitude AS DOUBLE)"))
    .withColumn("longitude_clean", F.expr("try_cast(longitude AS DOUBLE)"))
    .withColumn(
        "review_group",
        F.coalesce(
            F.concat(F.lit("cluster:"), clean_text(F.col("cluster_id"))),
            F.concat(F.lit("facility:"), F.col("facility_id")),
        ),
    )
)

from pyspark.sql.window import Window

source_row = F.to_json(
    F.struct(*[F.col(name) for name in sorted(facilities.columns)]),
    {"ignoreNullFields": "false"},
)

id_group = Window.partitionBy("facility_id")

clean_facilities = (
    clean_facilities
    .withColumn("_source_row", source_row)
    .dropDuplicates(["_source_row"])
    .withColumn("_id_count", F.count(F.lit(1)).over(id_group))
    .withColumn("original_facility_id", F.col("facility_id"))
    .withColumn(
        "facility_id_status",
        F.when(F.col("original_facility_id").isNull(), "missing_source_id")
        .when(F.col("_id_count") > 1, "reused_source_id")
        .otherwise("unique_source_id"),
    )
    .withColumn(
        "facility_id",
        F.when(
            F.col("facility_id_status") == "unique_source_id",
            F.col("original_facility_id"),
        ).otherwise(
            F.concat(F.lit("row:"), F.sha2(F.col("_source_row"), 256))
        ),
    )
    .withColumn(
        "review_group",
        F.coalesce(
            F.concat(F.lit("cluster:"), clean_text(F.col("cluster_id"))),
            F.concat(F.lit("source:"), F.col("original_facility_id")),
            F.concat(F.lit("facility:"), F.col("facility_id")),
        ),
    )
    .drop("_source_row", "_id_count")
)

for row in clean_facilities.groupBy("facility_id_status").count().collect():
    print(f"{row.facility_id_status}: {row['count']}")

postal_column_lookup = {
    name.lower().replace("_", ""): name for name in postal_directory.columns
}

region_columns = [
    postal_column_lookup[key]
    for key in ("statename", "state", "districtname", "district")
    if key in postal_column_lookup
]

if not region_columns:
    raise ValueError(
        f"No state or district column found. Available columns: {postal_directory.columns}"
    )

postal_reference = (
    postal_directory
    .select(
        clean_text(F.col("pincode")).alias("pin_code"),
        F.explode(F.array(*[clean_text(F.col(name)) for name in region_columns]))
        .alias("reference_region"),
    )
    .where(F.col("pin_code").rlike(r"^[1-9][0-9]{5}$"))
    .where(F.col("reference_region").isNotNull())
    .groupBy("pin_code")
    .agg(F.sort_array(F.collect_set("reference_region")).alias("pin_reference_regions"))
)

reported_region = normalize_place(F.col("reported_region"))

checked = (
    clean_facilities.join(postal_reference, "pin_code", "left")
    .withColumn(
        "pin_region_status",
        F.when(F.col("pin_code").isNull(), "missing_pin")
        .when(~F.col("pin_code").rlike(r"^[1-9][0-9]{5}$"), "invalid_pin_format")
        .when(F.col("pin_reference_regions").isNull(), "no_reference_match")
        .when(reported_region.isNull() | (reported_region == ""), "missing_reported_region")
        .when(
            F.exists("pin_reference_regions", lambda region: normalize_place(region) == reported_region),
            "region_match",
        )
        .otherwise("region_needs_review"),
    )
)

unique_source_id: 10077


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5293809160938213>, line 91
     87 for row in clean_facilities.groupBy("facility_id_status").count().collect():
     88     print(f"{row.facility_id_status}: {row['count']}")
     90 postal_column_lookup = {
---> 91     name.lower().replace("_", ""): name for name in postal_directory.columns
     92 }
     94 region_columns = [
     95     postal_column_lookup[key]
     96     for key in ("statename", "state", "districtname", "district")
     97     if key in postal_column_lookup
     98 ]
    100 if not region_columns:

File <command-5293809160938213>, line 8
      3 source_schema = (
      4     "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset"
      5 )
      7 facilities = spark.read.table(f"{source_schema}.facilities")
----> 8 postal_directory = spark.read.table(f"{source_schema}.india_post_pi

Select random data & check

In [0]:
%python
from pyspark.sql.window import Window

sample_per_tier = 10

latitude = F.col("latitude_clean")
longitude = F.col("longitude_clean")
coordinates_missing = (
    latitude.isNull() | longitude.isNull() | F.isnan(latitude) | F.isnan(longitude)
)

feature_rules = {
    "missing_facility_name": F.col("facility_name").isNull(),
    "missing_street_address": F.col("street_address").isNull(),
    "missing_or_invalid_coordinates": coordinates_missing,
    "coordinates_outside_india_screen": (
        ~coordinates_missing
        & ~(latitude.between(6, 38) & longitude.between(68, 98))
    ),
    "missing_pin": F.col("pin_region_status") == "missing_pin",
    "invalid_pin_format": F.col("pin_region_status") == "invalid_pin_format",
    "pin_absent_from_reference": F.col("pin_region_status") == "no_reference_match",
    "missing_reported_region": F.col("pin_region_status") == "missing_reported_region",
    "pin_region_needs_review": F.col("pin_region_status") == "region_needs_review",
}

feature_names = list(feature_rules)

features = checked.select(
    "*",
    *[
        F.coalesce(rule, F.lit(False)).cast("int").alias(name)
        for name, rule in feature_rules.items()
    ],
).withColumn(
    "quality_issues",
    F.filter(
        F.array(*[F.when(F.col(name) == 1, F.lit(name)) for name in feature_names]),
        lambda issue: issue.isNotNull(),
    ),
)

baseline = (
    features
    .withColumn(
        "automatic_tier",
        F.when(F.col("coordinates_outside_india_screen") == 1, "unreliable")
        .when(F.size("quality_issues") == 0, "credible")
        .otherwise("uncertain"),
    )
    .withColumn(
        "automatic_reason",
        F.when(
            F.col("automatic_tier") == "unreliable",
            "coordinates_outside_india_screen",
        )
        .when(F.col("automatic_tier") == "credible", "basic_checks_passed")
        .otherwise("basic_checks_need_review"),
    )
)

sample_order = Window.partitionBy("automatic_tier").orderBy(
    F.xxhash64("facility_id", F.lit(42)),
    F.col("facility_id"),
)

review_sample = (
    baseline
    .withColumn("sample_rank", F.row_number().over(sample_order))
    .where(F.col("sample_rank") <= sample_per_tier)
    .select(
        "automatic_tier", "sample_rank", "facility_id", "facility_name",
        "street_address", "city", "reported_region", "pin_code",
        "latitude_clean", "longitude_clean", "quality_issues", "source_urls",
    )
    .orderBy("automatic_tier", "sample_rank")
)

for row in review_sample.collect():
    print(
        f"\n[{row.automatic_tier} #{row.sample_rank}] "
        f"{row.facility_name} | {row.facility_id}\n"
        f"Address: {row.street_address}, {row.city}, {row.reported_region}, {row.pin_code}\n"
        f"Coordinates: {row.latitude_clean}, {row.longitude_clean}\n"
        f"Issues: {row.quality_issues}\n"
        f"Sources: {row.source_urls}"
    )


[credible #1] Astha hair clinic & Transplant Centre | bf879903-caed-4dce-ac44-72abd05094df
Address: 303/a , Sunlightplaza Complex, Near Balak Hanuman Temple, Pedak Road, Near Balak Hanuman Temple, Arya Nagar, Pedak Road, Pedak Road, Rajkot, Gujarat, 360003
Coordinates: 22.297271728515625, 70.80145263671875
Issues: []
Sources: ["https://www.facebook.com/313930132013170","https://www.justdial.com/Rajkot/Dr-Ashish-Dabhi-Astha-Hair-Clinic-Transplant-Centre-Near-Balak-Hanuman-Temple-Arya-Nagar-Pedak-Road/0281PX281-X281-170125065444-W7B3_BZDET/reviews","https://www.justdial.com/Rajkot/Astha-Hair-Clinic-Transplant-Center-Near-Balak-Hanuman-Temple-Pedak-Road/0281PX281-X281-140708074925-K9F7_BZDET","https://www.justdial.com/Rajkot/Dr-Ashish-Dabhi-Astha-Hair-Clinic-Transplant-Centre-Near-Balak-Hanuman-Temple-Arya-Nagar-Pedak-Road/0281PX281-X281-170125065444-W7B3_BZDET",null]

[credible #2] Pramida Dental & Faciomaxillary Hospital | 939b2ad3-2e65-4323-99f7-57d461dc4964
Address: House No 16-2-741

Data to check

In [0]:
%python
from pyspark.sql import functions as F

review_table = (
    review_sample
    .orderBy("automatic_tier", "sample_rank")
    .select(
        F.col("automatic_tier").alias("Auto Tier"),
        F.col("sample_rank").alias("No."),
        F.col("facility_name").alias("Facility"),
        F.col("city").alias("City"),
        F.col("reported_region").alias("Reported Region"),
        F.col("pin_code").alias("PIN Code"),
        F.col("street_address").alias("Street Address"),
        F.round("latitude_clean", 6).alias("Latitude"),
        F.round("longitude_clean", 6).alias("Longitude"),
        F.concat_ws(", ", "quality_issues").alias("Flagged Issues"),
        F.col("source_urls").alias("Source Links"),
        F.col("facility_id").alias("Facility ID"),
    )
)

display(review_table)

Auto Tier,No.,Facility,City,Reported Region,PIN Code,Street Address,Latitude,Longitude,Flagged Issues,Source Links,Facility ID
credible,1,Astha hair clinic & Transplant Centre,Rajkot,Gujarat,360003,"303/a , Sunlightplaza Complex, Near Balak Hanuman Temple, Pedak Road, Near Balak Hanuman Temple, Arya Nagar, Pedak Road, Pedak Road",22.297272,70.801453,,"[""https://www.facebook.com/313930132013170"",""https://www.justdial.com/Rajkot/Dr-Ashish-Dabhi-Astha-Hair-Clinic-Transplant-Centre-Near-Balak-Hanuman-Temple-Arya-Nagar-Pedak-Road/0281PX281-X281-170125065444-W7B3_BZDET/reviews"",""https://www.justdial.com/Rajkot/Astha-Hair-Clinic-Transplant-Center-Near-Balak-Hanuman-Temple-Pedak-Road/0281PX281-X281-140708074925-K9F7_BZDET"",""https://www.justdial.com/Rajkot/Dr-Ashish-Dabhi-Astha-Hair-Clinic-Transplant-Centre-Near-Balak-Hanuman-Temple-Arya-Nagar-Pedak-Road/0281PX281-X281-170125065444-W7B3_BZDET"",null]",bf879903-caed-4dce-ac44-72abd05094df
credible,2,Pramida Dental & Faciomaxillary Hospital,Malakpet,Telangana,500036,"House No 16-2-741/28, Plot No 38, Adjacent To Hanuman Temple, Near To Konark Diagnostic Centre, Andhra Bank Colony, Asmangadh",17.368481,78.517517,,"[""https://www.justdial.com/Hyderabad/Pramida-Dental-Faciomaxillary-Hospital-Adjacent-To-Hanuman-Temple-Near-To-Konark-Diagnostic-Centre-Malakpet/040PXX40-XX40-121102172328-J7H1_BZDET"",""https://www.facebook.com/pramidadentalhospitals/"",""https://www.pramidadentalhospitals.com/"",""https://www.facebook.com/100109219191271"",null]",939b2ad3-2e65-4323-99f7-57d461dc4964
credible,3,Kanpur Dental Clinic,Kanpur,Uttar Pradesh,208025,"Tilak Nagar Near Mr Brown Bakery, Geeta Nagar Road, Geeta Nagar, Kakadeo",26.484459,80.296776,,"[""https://www.lybrate.com/kannur/dental-clinics/fazal-ganj-kannur"",""https://www.indiamart.com/pandey-dental-clinic-kanpur/"",""https://www.sulekha.com/dentists/kanpur"",""https://www.facebook.com/1077454112391890"",""https://www.medindia.net/directories/hospitals/pandey-dental-clinic-kanpur-uttar-pradesh-55166.htm"",""http://www.dentistinkanpur.in/"",null]",7c6bdd31-7be8-46fa-a9fc-a7a3cc12788b
credible,4,GS Multispeciality Hospital,Basti,Uttar Pradesh,272001,"Malviya Road, Near Satyendra Hotal",26.799269,82.734375,,"[""https://ayushmancardhospitals.com/sri-brij-sewa-samiti-tb-sanatorium-mathura-uttar-pradesh/"",""https://www.policybazaar.com/network-hospitals/bajaj-allianz-hospitals-uttar_pradesh-basti/"",""https://www.justdial.com/Basti/Gs-Multi-Speciality-Hospital-Malviya-Road/9999P5542-5542-231005220708-J1Q5_BZDET"",""https://mapcarta.com/N7494190158"",""https://www.mappls.com/place-gs+multispeciality+hospital-malviya+road-rameshwarpuri-basti-uttar+pradesh-272001-KUVC8J@zdata=MjYuNzk0MTA2KzgyLjcyNjQyNisxNytLVVZDOEorK25yed"",""https://www.latlong.net/pois/in/basti/hospitals"",""https://www.facebook.com/101923982232826"",""https://www.facebook.com/102278082733320"",""https://dsinstitute.co.in/academic-detail/9"",""https://www.latlong.net/poi/prince-pharmacy-medical-centre-381779"",null]",775b8187-6b42-4107-a34a-ebfb24d193af
credible,5,Lakshya Homoeopathic Clinic,Haridwar,Uttarakhand,249402,"C-59, Shiv Mandir Rd, Near Mansi Beauty Parlour, C Cluster, Shivalik Nagar",29.934616,78.075516,,"[""https://ind.5bestincity.com/homeopathic-clinics-in-haridwar-uk"",""https://www.facebook.com/112950866267108"",""https://www.justdial.com/Haridwar/Lakshya-Homeopathic-Clinic-Near-Bsnl-Tower-Shivalik-Nagar/9999P1334-1334-181218155804-X9R3_BZDET"",""https://www.facebook.com/drgaurav.drpreeti/"",""https://www.addressguru.in/lakshya-homoeopathic-clinic"",null]",47d499a3-43d7-4ded-b7d2-07b290f64f99
credible,6,Ramana Maharishi Rangammal Hospital,Tiruvannamalai,Tamil Nadu,606603,"Shivanagar, Athyandal Village, Sri Ramanasaramam Post, Athiyandal",12.226064,79.025772,,"[""https://www.mappls.com/iygjg7"",""https://www.tnurbantree.tn.gov.in/thiruthangal/hospitals/"",""https://in.linkedin.com/company/sisu-hospital-india-ltd"",""http://www.onefivenine.com/india/villages/Tiruvan

Check

In [0]:
%python
review_rows = [
#1-10
 (
        "b8a5401f-42f1-422a-8cd9-686a15b4cb76",
        "conflict",
        "Aravind Eye Hospital is listed in Hyderabad, but the recorded coordinates (11.938235, 79.485443) are outside Hyderabad. Confirm the intended branch before correcting the address or coordinates. Source: https://aravind.org/contact-us/"
    ),
    (
        "06d5fb63-b2f1-4001-9c7c-a9fea7241dc3",
        "consistent",
        "Fortis Hospital, Gurugram matches the facility report for Sector 44, PIN 122002, and coordinates near (28.4572, 77.0728). Source: https://www.fortishealthcare.com/bio-medical-waste-report/annual-report/fortis-memorial-research-institute-gurgaon/online%20submission%20of%20bmw%20annual%20report%20to%20hspcb%20on%20form%20iv%20%28jan.01%2C%202022%20-dec.31%2C%202022%29"
    ),
    (
        "29b6dbe0-f471-4650-a488-10906b6ac873",
        "conflict",
        "The Fortis Hospital Anandapur record lists Tata Medical Center's website. Verify the street address and coordinates against the Fortis branch before using the row on a map. Sources: https://www.fortishealthcare.com/location/fortis-hospital-anandapur-kolkata ; https://tmckolkata.com/in/"
    ),
    (
        "3ac5ae50-9ce2-4de3-a13c-d95a4ef218ee",
        "conflict",
        "Wockhardt Hospital Nagpur is listed with a Fortis website and PIN 400078. Wockhardt's contact page lists its Ambazari Road location in Nagpur with PIN 440033. The recorded coordinates also need correction. Source: https://www.wockhardthospitals.com/contact-us/"
    ),
    (
        "59b6c976-2d44-4705-bcff-a83605983487",
        "conflict",
        "The street address and PIN 209217 match Rama Hospital's Mandhana branch in Kanpur, but the recorded coordinates (28.696606, 77.607498) are far from the branch location. Sources: https://ramahospital.com/contact/ ; https://www.latlong.net/poi/rama-medical-college-hospital-kanpur-nagar-382877"
    ),
    (
        "77a4e680-bf90-4cd0-b4b5-32bcdd7cf61c",
        "conflict",
        "The address opposite Mahamarg Bus Stop matches HCG Manavata Cancer Centre Unit 1 in Nashik. The hospital's directions link points to (19.989936, 73.783059), which conflicts with the recorded coordinates (21.105019, 79.101837). Source: https://www.hcghospitals.com/our-centres/hcg-manavata-cancer-centre-nashik-unit-1"
    ),
    (
        "88c2565a-8801-4899-9745-6070316eb745",
        "conflict",
        "The record lists PIN 562114. RajaRajeswari Medical College and Hospital lists its Mysore Road campus at Kambipura, Bengaluru, PIN 560074. Verify the coordinates against that campus before mapping. Source: https://www.rrmch.org/"
    ),
    (
        "8b4d0b3f-896c-4010-8b7e-b5ebd781fb29",
        "conflict",
        "Medanta's official address matches Sector 38, Gurugram, PIN 122001. The recorded coordinates (28.408033, 76.963745) are about 8 km from the mapped campus. Sources: https://www.medanta.org/hospitals-near-me/gurugram-hospital ; https://mapcarta.com/W348234206"
    ),
    (
        "d9bc3999-4523-4969-aac0-abd6bc415f2e",
        "insufficient",
        "Sumitra Hospital's name, Sector 35 address, and PIN 201301 are supported. The CSV PIN only needs whitespace removed. Published coordinates differ, so confirm the map pin before marking the location consistent. Sources: https://sumitrahospital.com/book-an-appointment/ ; https://www.latlong.net/poi/sumitra-nursing-home-gautam-buddha-nagar-404897 ; https://www.latlong.net/poi/sumitra-hospital-382387"
    ),
    (
        "e0fcd8ad-b818-4cef-975d-afdfd2e6d413",
        "conflict",
        "Government Medical College, Thiruvananthapuram lists PIN 695011. The CSV lists PIN 695009 and links to Jubilee Mission Medical College in Thrissur. Verify the address and coordinates against the intended campus. Sources: https://tmc.kerala.gov.in/ ; https://medicalcollege.jmmcri.org/contact-us/"
    ),
#11
    (
    "e0a0c7f8-b375-4fbe-997a-65ede0a721d1",
    "conflict",
    "VHM Hospitals is listed in Saligramam, Chennai (PIN 600093), confirmed by its website and Bajaj Finserv Health. The dataset's coordinates (13.028790, 80.256729) are about 6 km east of Saligramam, which Wikipedia places at (13.0481, 80.2014); use the latter as an approximate correction. Sources: https://www.vhmhospitals.com/ ; https://www.bajajfinservhealth.in/hospitals/chennai/vhm-hospital-pvt-ltd-k-k-nagar ; https://en.wikipedia.org/wiki/Saligramam"
),
#12
(
    "17fd3d32-fbc5-47be-bd65-7763bd967d69",
    "consistent",
    "All proven valid. Sources: https://www.ssmfpune.in/our-centres.html ; https://www.ssmfpune.in/"
),
#13-17
    (
        "78bed667-b16e-43c1-8d82-c82891f23aa8",
        "consistent",
        "All proven valid. Sources: https://vidyahospitalraipur.com/ ; https://bajajfinservhealth.in/hospitals/raipur/vidya-hospital-shankar-nagar ; https://www.myhospitalnow.com/hospitals/vidya-hospital-and-kidney-center"
    ),
    (
        "bafdfd43-51a2-413a-b0b9-bb19275c2833",
        "consistent",
        "All proven valid. Sources: http://laxmidentalclinic.co.in/"
    ),
    (
        "0ee89a39-cdb7-4261-a644-36f0edfd5a4e",
        "conflict",
        "Shrihari Imaging Centre is listed at Pramukh Mastana Complex, Reliance Cross Road, Kudasan, Gandhinagar, with PIN 382009. Bajaj Finserv Health confirms the address, and the coordinates (23.185053, 72.63205) fall in Kudasan, but other businesses in Kudasan use PIN 382421. The PIN should be corrected to 382421. Sources: https://bajajfinservhealth.in/lab-tests/diagnostic-centers/Shrihari_Imaging_Centre ; https://labs.lalpathlabs.com/dr-lal-pathlabs-patient-service-centre-diagnostic-centre-kudasan-gandhinagar-473733/Offers"
    ),
    (
        "e6f3ae66-644c-40b3-a2d5-90c9fd0a62a7",
        "conflict",
        "Smart City Hospital is listed at Bharat Scout and Guide Building, Near Depot Square, Shyamla Hills, Bhopal, with PIN 462013. Bajaj Finserv Health, HexaHealth and insurer network lists give the same address with PIN 462003, and the coordinates (23.232056, 77.387993) fall in Shyamla Hills. The PIN should be corrected to 462003. Sources: https://www.bajajfinservhealth.in/hospitals/bhopal/smart-city-hospital ; https://www.hexahealth.com/bhopal/hospital/smart-city-hospital-tt-nagar"
    ),
#18 19
(
    "e5efd386-a9b5-449c-8bae-c295742d7085",
    "conflict",
    "The clinic website matches the Munirka street address but lists PIN 110067, while the dataset lists 67. The exact building coordinates were not independently verified. Source: https://drkumarshomeopathyandmentalhealth.com/about",
),
(
    "0201050b-de4b-46ab-9425-68147e690173",
    "conflict",
    "The dataset lists street number 57-7-25 and PIN 10. The hospital contact page lists 57-7-23/1, and the D&B listing corroborates that street number and gives the full PIN 520010. Sources: https://www.vedakidneyhospital.in/contact.html ; https://www.dnb.com/business-directory/company-profiles/veda-hospital--kidney-centre%C2%A0.3b24e8d89fb3f867a3f3cf1195ec4c65",
),
#20
(
    "8403e19b-c0fe-43c9-8c1e-fa313e832364",
    "conflict",
    "Aishal Medicity is listed in Kanhangad, Kerala, India, with PIN code 671329 in the dataset. However, external sources report PIN code 671315 for Aishal Medicity at North Kottachery, Kanhangad. This creates a conflict between the dataset and external evidence. The recorded coordinates (12.327222, 75.08551) are geographically plausible, but the postal code requires correction or further verification. Sources: https://safewaytpa.in/Documents/NIVA_Excluded_List_Revised2.pdf ; https://aishalmedicity.in/ ; https://www.google.com/search?kgmid=%2Fg%2F11sr7qfzkm"
),
#21:
(
    "86a33104-2959-4ddf-aeda-4e6f42b2395d",
    "conflict",
    "The Family Tree Hospital is listed in Tirupati, Andhra Pradesh, India, with PIN code 517501. However, the dataset reports coordinates (32.962585, 7.479839), which fall outside India. Google Maps identifies The Family Tree Hospital near Padmavathi Park, Bairagipatteda, Tirupati, with coordinates (13.617697, 79.422627). The facility name, city, and PIN code are consistent, but the dataset's latitude and longitude conflict with the mapped location. The Google Maps coordinates are proposed corrections that should be retained with their source. Sources: https://www.google.com/maps/search/?api=1&query=13.617697%2C79.422627 ; https://thefamilytreehospital.com/"
),
#22
(
    "b288467c-117e-47fe-81b5-7bb8dcf494b3",
    "conflict",
    "HZB Arogyam Multispeciality Hospital is listed in Hazaribagh, Jharkhand, India, with PIN code 825301. However, the dataset reports coordinates (46.073227, 106.171875), which fall outside India. Google Maps identifies the hospital near District Board Chowk, Zila Parishad Bhawan, C.T.S. Colony, Hazaribagh, at coordinates (23.999175, 85.365332). The facility name, city, state, and PIN code are consistent, but the dataset's coordinates clearly conflict with the mapped location. The Google Maps coordinates are proposed corrections based on the map screenshot. Sources: https://www.google.com/maps/search/?api=1&query=23.999175%2C85.365332 ; https://myhospitalnow.com/hospitals/hzb-arogyam-multispeciality-hospital"
),
#23-26
   (
        "fac0cc70-81a9-462d-bced-5ca40257b027",
        "conflict",
        "Sanjivani Multi Speciality Hospital is listed in Chengannur, Alappuzha, Kerala, with PIN code 690509. The dataset coordinates (59.949684, -38.262634) fall outside India. Google Maps identifies a hospital with the same name in Kollakadavu, Kerala, with coordinates (9.403596, 76.585305) and the same PIN code 690509. This supports a geographic coordinate conflict. A second screenshot shows (9.775506, 81.357425), but that point is not on the identified hospital and should not be used as the correction. Sources: Google Maps screenshot showing Sanjivani Multi Speciality Hospital at Kollakadavu; https://smsh.in/ ; https://www.google.com/maps/search/?api=1&query=9.403596%2C76.585305"
    ),
    (
        "7c6e6ca4-d07f-4613-a926-47d5ae7a2afe",
        "conflict",
        "Cardia Health Care is listed in Noida, Uttar Pradesh, with PIN code 201305. The dataset coordinates (7.710991, 109.6875) fall outside India. Google Maps identifies a similarly named Cardia Healthcare at Eranhipalam, Kozhikode, Kerala, with coordinates (11.277291, 75.784830), but the listing describes a surgical supply store rather than a hospital. This is not sufficient evidence that the Kozhikode business and the Noida facility are the same entity. The dataset has a coordinate conflict, while the correct facility identity and location remain unverified. Sources: https://www.google.com/maps/search/?api=1&query=11.277291%2C75.784830 ; https://www.meerutonline.in/trauma-accident-and-emergency-care/30724"
    ),
    (
        "515fb95f-d589-418f-b510-d8bf7438f884",
        "conflict",
        "Cura Imaging & Gastro Clinic is listed in Nagpur, Maharashtra, with PIN code 411010. The dataset coordinates (2.952934, 41.387157) fall outside India. Google Maps shows a listing for Cura Clinic near Lokmat Square, Ramdaspeth, Nagpur, with coordinates (21.136878, 79.078411) and PIN code 440012. The address closely matches the dataset's Ramdaspeth and Lokmat Square details, suggesting that this may be the same facility. Both the coordinates and PIN code conflict with the dataset. The business identity should be confirmed before automatically applying the replacement values. Sources: https://www.google.com/maps/search/?api=1&query=21.136878%2C79.078411 ; https://www.nagpuronline.in/business/cura-imaging-and-gastro-clinic-57790"
    ),
    (
        "364dae5c-b868-450a-9e4e-e6705852c229",
        "conflict",
        "Krishna Hospital Multispeciality is listed in Lucknow, Uttar Pradesh, with PIN code 226021. The dataset coordinates (-81.706299, 26.95311) fall outside India. Google Maps identifies Krishna Hospital Multispeciality near Sitapur Road, Tadikhana, Lucknow, with coordinates (26.907921, 80.938267). The mapped address closely matches the dataset's street address, but Google Maps lists PIN code 226020 rather than 226021. Therefore, the dataset has a clear coordinate conflict and a postal-code discrepancy requiring verification. Sources: https://www.google.com/maps/search/?api=1&query=26.907921%2C80.938267 ; https://wanderlog.com/place/details/15122546/krishna-hospital-multispeciality"
    )
]

allowed_statuses = {"consistent", "conflict", "insufficient"}
review_ids = [facility_id for facility_id, _, _ in review_rows]

if len(review_ids) != len(set(review_ids)):
    raise ValueError("Each facility ID may appear only once in review_rows.")

if any(
    not facility_id.strip() or status not in allowed_statuses or not note.strip()
    for facility_id, status, note in review_rows
):
    raise ValueError("Each review needs a facility ID, a valid status, and an evidence note.")

reviews = spark.createDataFrame(
    review_rows,
    "facility_id string, review_status string, review_note string",
)

unknown_ids = reviews.join(
    baseline.select("facility_id"),
    "facility_id",
    "left_anti",
)

if unknown_ids.take(1):
    raise ValueError("At least one reviewed facility ID was not found in the source table.")

auto + people check

In [0]:
%python
from pyspark.sql import functions as F

reviews = spark.createDataFrame(
    review_rows,
    "facility_id string, review_status string, review_note string",
)

training_data = (
    baseline.select("facility_id", "review_group", *feature_names)
    .join(reviews, "facility_id", "inner")
    .where(F.col("review_status").isin("consistent", "conflict"))
    .withColumn("target", (F.col("review_status") == "conflict").cast("int"))
    .orderBy("facility_id")
    .toPandas()
)

print(f"Consistent records: {(training_data['target'] == 0).sum()}")
print(f"Conflict records: {(training_data['target'] == 1).sum()}")

Consistent records: 4
Conflict records: 20


logistic regression

In [0]:
%python
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold

model = None
X = training_data[feature_names]
y = training_data["target"]
groups = training_data["review_group"]

groups_per_class = training_data.groupby("target")["review_group"].nunique()

if len(groups_per_class) < 2 or groups_per_class.min() < 2:
    raise ValueError(
        "A grouped evaluation requires both consistent and conflict labels, "
        "with at least two distinct facility groups per label."
    )

splitter = StratifiedGroupKFold(
    n_splits=min(4, int(groups_per_class.min())),
    shuffle=True,
    random_state=42,
)

split = next(
    (
        (train_idx, test_idx)
        for train_idx, test_idx in splitter.split(X, y, groups)
        if y.iloc[train_idx].nunique() == 2
        and y.iloc[test_idx].nunique() == 2
        and X.iloc[train_idx].nunique().max() > 1
    ),
    None,
)

if split is None:
    raise ValueError(
        "No usable grouped split was found. "
        "Review more facility groups with different issue patterns."
    )

train_idx, test_idx = split
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X.iloc[train_idx], y.iloc[train_idx])

predictions = model.predict(X.iloc[test_idx])
tn, fp, fn, tp = confusion_matrix(
    y.iloc[test_idx], predictions, labels=[0, 1]
).ravel()

print(f"Training records: {len(train_idx)}")
print(f"Test records: {len(test_idx)}")
print(f"Correctly identified consistent records: {tn}")
print(f"False conflict alerts: {fp}")
print(f"Missed conflicts: {fn}")
print(f"Detected conflicts: {tp}")

for name, weight in zip(feature_names, model.coef_[0]):
    print(f"{name}: {weight:+.4f}")

Training records: 18
Test records: 6
Correctly identified consistent records: 1
False conflict alerts: 2
Missed conflicts: 2
Detected conflicts: 1
missing_facility_name: +0.0000
missing_street_address: +0.0000
missing_or_invalid_coordinates: +0.0000
coordinates_outside_india_screen: +0.9860
missing_pin: +0.0000
invalid_pin_format: +0.6473
pin_absent_from_reference: +0.5673
missing_reported_region: +0.0000
pin_region_needs_review: +0.0000


In [0]:
%python
from pyspark.sql import functions as F

output_schema = "workspace.default"

score_inputs = baseline.select("facility_id", *feature_names).toPandas()
conflict_scores = model.predict_proba(score_inputs[feature_names])[:, 1]

model_scores = spark.createDataFrame(
    [
        (facility_id, float(score))
        for facility_id, score in zip(score_inputs["facility_id"], conflict_scores)
    ],
    "facility_id string, model_conflict_score double",
)

final_facilities = (
    baseline.join(model_scores, "facility_id", "left")
    .join(reviews, "facility_id", "left")
    .withColumn(
        "record_quality_reason",
        F.when(F.col("review_status") == "conflict", "manual_review_conflict")
        .when(
            F.col("coordinates_outside_india_screen") == 1,
            "coordinates_outside_india_screen",
        )
        .when(F.col("review_status") == "insufficient", "manual_review_insufficient")
        .when(
            (F.col("review_status") == "consistent")
            & (F.col("missing_facility_name") == 0)
            & (F.col("missing_or_invalid_coordinates") == 0),
            "manual_review_consistent",
        )
        .otherwise(F.col("automatic_reason")),
    )
    .withColumn(
        "record_quality_tier",
        F.when(
            F.col("record_quality_reason").isin(
                "manual_review_conflict", "coordinates_outside_india_screen"
            ),
            "unreliable",
        )
        .when(
            F.col("record_quality_reason").isin(
                "manual_review_consistent", "basic_checks_passed"
            ),
            "credible",
        )
        .otherwise("uncertain"),
    )
)

for tier in ("credible", "uncertain", "unreliable"):
    table_name = f"{output_schema}.cleanfacility_{tier}"
    (
        final_facilities.where(F.col("record_quality_tier") == tier)
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    print(f"Saved {table_name}")

Saved workspace.default.cleanfacility_credible
Saved workspace.default.cleanfacility_uncertain
Saved workspace.default.cleanfacility_unreliable


In [0]:
%sql
WITH counts AS (
    SELECT 'credible' AS tier, COUNT(*) AS records
    FROM workspace.default.cleanfacility_credible
    UNION ALL
    SELECT 'uncertain', COUNT(*)
    FROM workspace.default.cleanfacility_uncertain
    UNION ALL
    SELECT 'unreliable', COUNT(*)
    FROM workspace.default.cleanfacility_unreliable
)
SELECT tier, records FROM counts
UNION ALL
SELECT 'total', SUM(records) FROM counts;

tier,records
total,10077
credible,9244
uncertain,813
unreliable,20


Clean data

In [0]:
%python
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType

tiers = ("credible", "uncertain", "unreliable")
source_tables = {
    tier: spark.read.table(f"workspace.default.cleanfacility_{tier}")
    for tier in tiers
}

facility_data = reduce(
    lambda left, right: left.unionByName(right),
    [
        table.withColumn("matrix_tier", F.lit(tier))
        for tier, table in source_tables.items()
    ],
)

matrix_rows = facility_data.select("matrix_tier", "facility_id")
row_counts = {
    row["matrix_tier"]: row["count"]
    for row in matrix_rows.groupBy("matrix_tier").count().collect()
}

numeric_fields = [
    ("location", "latitude_clean", "latitude"),
    ("location", "longitude_clean", "longitude"),
    ("staffing", "reported_doctor_count", "reported_doctor_count"),
    ("facilities", "capacity", "reported_capacity"),
]

category_fields = [
    ("location", "city"),
    ("location", "reported_region"),
    ("location", "pin_code"),
    ("location", "pin_region_status"),
]

list_fields = [
    ("staffing", "specialties"),
    ("facilities", "equipment"),
    ("facilities", "capability"),
    ("facilities", "procedure"),
]

def clean_label(value):
    value = F.lower(F.trim(F.regexp_replace(value, r"\s+", " ")))
    return F.when((value != "") & (value != "null"), value)

def list_values(field):
    column = F.col(field)
    if isinstance(facility_data.schema[field].dataType, ArrayType):
        return column.cast("array<string>")

    text = F.trim(column.cast("string"))
    return F.when(
        text.startswith("["),
        F.from_json(text, "array<string>"),
    ).otherwise(F.array(text))

def numeric_entries(category, field, feature):
    value = F.expr(f"try_cast(`{field}` AS DOUBLE)")
    return (
        facility_data.select(
            "matrix_tier",
            "facility_id",
            F.lit(category).alias("category"),
            F.lit(feature).alias("feature"),
            value.alias("value"),
        )
        .where(F.col("value").isNotNull() & ~F.isnan("value"))
    )

def label_entries(category, field, values):
    return (
        facility_data.select(
            "matrix_tier",
            "facility_id",
            F.lit(category).alias("category"),
            F.explode(values).alias("label"),
        )
        .withColumn("label", clean_label(F.col("label")))
        .where(F.col("label").isNotNull())
        .select(
            "matrix_tier",
            "facility_id",
            "category",
            F.concat(F.lit(f"{field}="), F.col("label")).alias("feature"),
            F.lit(1.0).alias("value"),
        )
    )

matrix_entries = reduce(
    lambda left, right: left.unionByName(right),
    [numeric_entries(*spec) for spec in numeric_fields]
    + [
        label_entries(category, field, F.array(F.col(field).cast("string")))
        for category, field in category_fields
    ]
    + [
        label_entries(category, field, list_values(field))
        for category, field in list_fields
    ],
).dropDuplicates(["matrix_tier", "facility_id", "category", "feature"])

feature_catalog = (
    matrix_entries.select("category", "feature")
    .unionByName(
        spark.createDataFrame(
            [(category, feature) for category, _, feature in numeric_fields],
            "category string, feature string",
        )
    )
    .distinct()
)

feature_counts = {
    row["category"]: row["count"]
    for row in feature_catalog.groupBy("category").count().collect()
}

categories = ("location", "staffing", "facilities")
matrix_width = sum(feature_counts.get(category, 0) for category in categories)

matrices = {
    tier: {
        category: matrix_entries.where(
            (F.col("matrix_tier") == tier) & (F.col("category") == category)
        ).select("facility_id", "feature", "value")
        for category in categories
    }
    for tier in tiers
}

for tier in tiers:
    rows = row_counts.get(tier, 0)
    print(f"\ncleanfacility_{tier}: {rows:,} x {len(source_tables[tier].columns):,}")
    for category in categories:
        columns = feature_counts.get(category, 0)
        print(f"  {category} matrix: {rows:,} x {columns:,}")
    print(f"  Combined feature matrix: {rows:,} x {matrix_width:,}")

print(f"\nAll tiers combined: {sum(row_counts.values()):,} x {matrix_width:,}")


cleanfacility_credible: 9,244 x 83
  location matrix: 9,244 x 5,118
  staffing matrix: 9,244 x 2,704
  facilities matrix: 9,244 x 281,755
  Combined feature matrix: 9,244 x 289,577

cleanfacility_uncertain: 813 x 83
  location matrix: 813 x 5,118
  staffing matrix: 813 x 2,704
  facilities matrix: 813 x 281,755
  Combined feature matrix: 813 x 289,577

cleanfacility_unreliable: 20 x 83
  location matrix: 20 x 5,118
  staffing matrix: 20 x 2,704
  facilities matrix: 20 x 281,755
  Combined feature matrix: 20 x 289,577

All tiers combined: 10,077 x 289,577


In [0]:
%python
from functools import reduce

from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType

tiers = ("credible", "uncertain", "unreliable")
keys = ["matrix_tier", "facility_id"]

source_tables = {
    tier: spark.read.table(f"workspace.default.cleanfacility_{tier}")
    for tier in tiers
}

facility_data = reduce(
    lambda left, right: left.unionByName(right),
    [
        table.withColumn("matrix_tier", F.lit(tier))
        for tier, table in source_tables.items()
    ],
)

def schema_key(name):
    return "".join(character for character in name.lower() if character.isalnum())


column_lookup = {schema_key(name): name for name in facility_data.columns}


def read_text_items(field):
    column = F.col(field)
    empty = F.array().cast("array<string>")

    if isinstance(facility_data.schema[field].dataType, ArrayType):
        items = column.cast("array<string>")
    else:
        text = F.trim(column.cast("string"))
        items = F.when(
            text.startswith("["),
            F.coalesce(F.from_json(text, "array<string>"), F.array(text)),
        ).otherwise(F.array(text))

    return F.filter(
        F.coalesce(items, empty),
        lambda item: item.isNotNull()
        & ~F.lower(F.trim(item)).isin("", "null", "none", "n/a"),
    )


count_fields = {
    "staffing": {
        "doctor_count": ("reported_doctor_count", "numberDoctors", "doctor_count"),
        "nurse_count": ("reported_nurse_count", "numberNurses", "nurse_count"),
        "specialist_count": ("reported_specialist_count", "numberSpecialists"),
        "midwife_count": ("reported_midwife_count", "numberMidwives"),
        "technician_count": ("reported_technician_count", "numberTechnicians"),
        "total_staff_count": ("reported_staff_count", "numberStaff", "total_staff_count"),
    },
    "facilities": {
        "reported_capacity": ("capacity", "reported_capacity"),
        "bed_count": ("reported_bed_count", "numberBeds", "bed_count"),
        "icu_bed_count": ("reported_icu_bed_count", "numberIcuBeds", "icu_bed_count"),
        "operating_room_count": ("reported_operating_room_count", "numberOperatingRooms"),
    },
}

matrix = facility_data
numeric_features = {"staffing": [], "facilities": []}

for category, definitions in count_fields.items():
    for feature, aliases in definitions.items():
        fields = dict.fromkeys(
            column_lookup[schema_key(alias)]
            for alias in aliases
            if schema_key(alias) in column_lookup
        )
        if not fields:
            continue

        values = []
        for field in fields:
            value = F.expr(f"try_cast(`{field}` AS DOUBLE)")
            values.append(F.when((value >= 0) & (value < float("inf")), value))

        matrix = matrix.withColumn(feature, F.coalesce(*values))
        numeric_features[category].append(feature)

equipment_fields = ("equipment",)
service_fields = ("capability", "procedure", "description")
specialty_fields = ("specialties",)

hours = (
    r"(?:24\s*(?:[/x×-]\s*7|hours?\s*/\s*7)|7\s*[/x×-]\s*24|"
    r"24[- ]hours?(?:\s+a\s+day)?|24\s+hours?\s+daily|"
    r"(?:round|around)[- ]the[- ]clock)"
)
emergency = (
    r"(?:(?:accident\s+(?:and|&)\s+)?emergency\s+"
    r"(?:care|services?|department|room|unit)|a\s*(?:&|and)\s*e|er)"
)
dental_emergency = (
    r"(?:dental\s+emergenc(?:y|ies)|emergency\s+dent(?:al|istry))"
    r"(?:\s+(?:care|services?))?"
)
connector = r"[\s,:()\[\]-]+(?:(?:is|are|available|open|operates?|operating)\s+)*"


def continuous_service(service):
    return (
        rf"\b{hours}(?:{connector}{hours})*{connector}{service}\b"
        rf"|\b{service}{connector}{hours}\b"
    )


general_emergency = (
    rf"^(?!.*\b{dental_emergency}\b).*?"
    rf"(?:{continuous_service(emergency)}"
    rf"|\b{hours}{connector}emergency\s*[.!]?$"
    rf"|^\s*emergency{connector}{hours}\s*[.!]?$)"
)

rules = [
    ("staffing", "specialty_cardiology", specialty_fields,
     r"\b(?:cardiology|cardiovascular medicine)\b"),
    ("staffing", "specialty_nephrology", specialty_fields,
     r"\b(?:nephrology|renal medicine)\b"),
    ("staffing", "specialty_pediatrics", specialty_fields,
     r"\b(?:paediatrics|pediatrics|paediatric medicine|pediatric medicine)\b"),
    ("staffing", "specialty_orthopedics", specialty_fields,
     r"\b(?:orthopaedics|orthopedics|orthopaedic surgery|orthopedic surgery)\b"),
    ("staffing", "specialty_ophthalmology", specialty_fields,
     r"\b(?:ophthalmology|ophthalmic medicine)\b"),
    ("staffing", "specialty_dermatology", specialty_fields,
     r"\bdermatology\b"),
    ("staffing", "specialty_neurology", specialty_fields,
     r"\bneurology\b"),
    ("staffing", "specialty_obstetrics", specialty_fields,
     r"\bobstetrics\b"),
    ("staffing", "specialty_gynecology", specialty_fields,
     r"\b(?:gynaecology|gynecology)\b"),

    ("facilities", "emergency_24_7", service_fields, general_emergency),
    ("facilities", "dental_emergency_24_7", service_fields,
     continuous_service(dental_emergency)),
    ("facilities", "mri_equipment", equipment_fields,
     r"\b(?:mri|magnetic resonance imaging)(?:\s+(?:scanners?|machines?|equipment))?\b"),
    ("facilities", "mri_scan", service_fields,
     r"\b(?:mri|magnetic resonance imaging)\s+(?:scans?|scanning|services?)\b"
     r"|^\s*magnetic resonance imaging\s*\.?$"),
    ("facilities", "ct_scanner", equipment_fields,
     r"\b(?:ct|cat|computed tomography)\s+(?:scanners?|machines?|equipment)\b"),
    ("facilities", "ct_scan", service_fields,
     r"\b(?:ct|cat|computed tomography)\s+(?:scans?|scanning|services?)\b"),
    ("facilities", "blood_draw", service_fields,
     r"\b(?:blood draws?|phlebotomy|ven[ei]puncture|"
     r"blood (?:sample|specimen) collection)\b"),
    ("facilities", "dialysis_machine", equipment_fields,
     r"\b(?:dialysis|ha?emodialysis)\s+(?:machines?|equipment)\b"),
    ("facilities", "hemodialysis", service_fields,
     r"\bha?emodialysis\b(?!\s+(?:machine|equipment))"),
    ("facilities", "peritoneal_dialysis", service_fields,
     r"\bperitoneal dialysis\b"),
    ("facilities", "ventilator", equipment_fields,
     r"\bventilators?\b"),
    ("facilities", "ultrasound_equipment", equipment_fields,
     r"\b(?:ultrasound|ultrasonography|sonography)\b"),
    ("facilities", "ultrasound_scan", service_fields,
     r"\b(?:ultrasonography|sonography|ultrasound\s+(?:scans?|scanning|imaging))\b"),
    ("facilities", "xray_equipment", equipment_fields,
     r"\bx[- ]?ray\s+(?:machines?|equipment|systems?)\b"),
    ("facilities", "nicu", service_fields,
     r"\b(?:nicu|neonatal intensive care units?)\b"),
    ("facilities", "icu", service_fields,
     r"\b(?:general intensive care unit|adult intensive care unit|adult icu)\b"
     r"|^\s*(?:icu|intensive care unit)\s*\.?$"),
    ("facilities", "cesarean_delivery", service_fields,
     r"\b(?:c[- ]sections?|(?:caesarean|cesarean)\s+(?:sections?|deliver(?:y|ies)))\b"),
]

text_fields = [
    field for field in dict.fromkeys((*equipment_fields, *service_fields, *specialty_fields))
    if field in facility_data.columns
]

text_records = reduce(
    lambda left, right: left.unionByName(right),
    [
        facility_data.select(*keys, F.explode(read_text_items(field)).alias("source_text"))
        .withColumn("source_field", F.lit(field))
        for field in text_fields
    ],
).withColumn(
    "normalized_text",
    F.trim(
        F.regexp_replace(
            F.regexp_replace(F.lower("source_text"), r"[‐‑–—]", "-"),
            r"\s+",
            " ",
        )
    ),
)

review_pattern = (
    r"\b(?:no|not|without|unavailable|closed|planned|proposed|"
    r"coming soon|out of service|off[- ]site|third[- ]party|"
    r"referr\w*|previously|formerly|discontinued)\b"
)

normalized_text = F.col("normalized_text")
raw_matches = F.array(*[
    F.when(
        F.col("source_field").isin(*fields) & normalized_text.rlike(pattern),
        F.struct(F.lit(category).alias("category"), F.lit(feature).alias("feature")),
    )
    for category, feature, fields, pattern in rules
])

classified = text_records.withColumn(
    "matches",
    F.filter(raw_matches, lambda match: match.isNotNull()),
)

feature_evidence = (
    classified.withColumn("match", F.explode("matches"))
    .select(
        *keys,
        F.col("match.category").alias("category"),
        F.col("match.feature").alias("feature"),
        "source_field",
        "source_text",
        normalized_text.rlike(review_pattern).cast("int").alias("review_flag"),
    )
    .groupBy(*keys, "category", "feature")
    .agg(
        F.sort_array(
            F.collect_set(
                F.struct(
                    F.col("source_field").alias("field"),
                    F.col("source_text").alias("text"),
                )
            )
        ).alias("evidence"),
        F.max("review_flag").alias("review_flag"),
    )
    .withColumn(
        "status",
        F.when(F.col("review_flag") == 1, "needs_review").otherwise("reported"),
    )
)

unmapped_terms = (
    classified.where(F.size("matches") == 0)
    .groupBy("source_field", "normalized_text")
    .agg(
        F.countDistinct(F.struct(*keys)).alias("facility_count"),
        F.min("source_text").alias("example_text"),
    )
    .orderBy(F.desc("facility_count"), "source_field", "normalized_text")
)

observed_features = {
    (row["category"], row["feature"])
    for row in feature_evidence.select("category", "feature").distinct().collect()
}

evidence_by_facility = feature_evidence.groupBy(*keys).agg(
    F.map_from_entries(
        F.collect_list(
            F.struct(
                F.col("feature").alias("key"),
                F.struct("status", "evidence").alias("value"),
            )
        )
    ).alias("feature_details")
)

matrix = matrix.join(evidence_by_facility, keys, "left")
empty_evidence = F.array().cast("array<struct<field:string,text:string>>")
standard_features = {
    category: list(features) for category, features in numeric_features.items()
}
feature_columns = []

for category, feature, _, _ in rules:
    if (category, feature) not in observed_features:
        continue

    detail = F.element_at(F.col("feature_details"), F.lit(feature))
    status = F.coalesce(detail["status"], F.lit("unknown"))
    feature_columns.extend([
        F.when(status == "reported", F.lit(1.0)).alias(feature),
        status.alias(f"{feature}_status"),
        F.coalesce(detail["evidence"], empty_evidence).alias(f"{feature}_evidence"),
    ])
    standard_features[category].append(feature)

staffing_features = standard_features["staffing"]
facility_features = standard_features["facilities"]

facility_matrix = (
    matrix.select("*", *feature_columns)
    .drop("feature_details")
    .withColumn("distance_km", F.lit(None).cast("double"))
    .withColumn("distance_vector", F.array("distance_km"))
    .withColumn("staffing_vector", F.array(*staffing_features).cast("array<double>"))
    .withColumn("facilities_vector", F.array(*facility_features).cast("array<double>"))
)

print(
    f"Standard features: {1 + len(staffing_features) + len(facility_features)} "
    f"(distance: 1, staffing: {len(staffing_features)}, "
    f"facilities and services: {len(facility_features)})"
)
print("\n".join(f"{i}. {name}" for i, name in enumerate(facility_features, start=1)))
display(facility_matrix.select("facility_id", "facility_name", "matrix_tier", *staffing_features, *facility_features).orderBy("matrix_tier", "facility_name"))

Standard features: 29 (distance: 1, staffing: 10, facilities and services: 18)
1. reported_capacity
2. emergency_24_7
3. dental_emergency_24_7
4. mri_equipment
5. mri_scan
6. ct_scanner
7. ct_scan
8. blood_draw
9. dialysis_machine
10. hemodialysis
11. peritoneal_dialysis
12. ventilator
13. ultrasound_equipment
14. ultrasound_scan
15. xray_equipment
16. nicu
17. icu
18. cesarean_delivery


facility_id,facility_name,matrix_tier,doctor_count,specialty_cardiology,specialty_nephrology,specialty_pediatrics,specialty_orthopedics,specialty_ophthalmology,specialty_dermatology,specialty_neurology,specialty_obstetrics,specialty_gynecology,reported_capacity,emergency_24_7,dental_emergency_24_7,mri_equipment,mri_scan,ct_scanner,ct_scan,blood_draw,dialysis_machine,hemodialysis,peritoneal_dialysis,ventilator,ultrasound_equipment,ultrasound_scan,xray_equipment,nicu,icu,cesarean_delivery
dc80c270-ec41-4542-89d7-ebe31e8407f8,"""BA"" Smt. Lilaben Chimanlal Parikh Cancer Centre",credible,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1.0,1.0,null,null,null,null,null,1.0,1.0,1.0,null,null,null
b4e0b960-ad16-4c18-b756-ed42ea6a9f39,21st Century Hospitals,credible,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1.0,null,null
bf4a4444-50f4-40b6-b4c3-078b459ef4d6,24x7 Rudraksha Multispeciality Hospitals,credible,null,null,null,null,null,null,null,null,null,null,60.0,null,null,null,null,1.0,null,null,1.0,null,null,null,1.0,1.0,1.0,1.0,null,null
d55564b5-5fe5-4ca1-b090-20e7fe58a31a,32 Dental Care Center,credible,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1.0,null,null,null
8992ef2b-0177-45b6-b1a2-391cf9419203,32 Diamonds Dental Clinic,credible,11.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1.0,null,null,null
39d1611b-a69f-4a5f-8818-8aae818b9e5e,32 Friends Dental Clinic,credible,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
21d2e77a-acb4-4894-b64b-e56466278ff2,32 Gems Dental Clinic,credible,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
87aa7bed-6884-43fe-ba2d-178de4bf39ac,32 Pearls Dental Clinic,credible,7.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
377a6a7c-2840-46c9-8867-4653624d8ae6,32 Pearls Dental Clinic,credible,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
551018b9-c62f-466a-aec1-8735a43a1d4a,32 Pearls Dental Clinic,credible,2.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


User input：longitude latitude, and what AI tells the user what 诊室 and facilities needed


In [0]:

%python

# Create user input fields
dbutils.widgets.text(
    "patient_latitude",
    "",
    "1. Patient Latitude"
)

dbutils.widgets.text(
    "patient_longitude",
    "",
    "2. Patient Longitude"
)

dbutils.widgets.text(
    "required_features",
    "",
    "3. Required Medical Features"
)

print("Please enter your location and required medical features in the widgets above.")


Please enter your location and required medical features in the widgets above.


In [0]:

%python
from pyspark.sql import functions as F

patient_lat = float(dbutils.widgets.get("patient_latitude"))
patient_lon = float(dbutils.widgets.get("patient_longitude"))

required_features = [
    x.strip().lower()
    for x in dbutils.widgets.get("required_features").split(",")
    if x.strip()
]

print("Patient location:", patient_lat, patient_lon)
print("Required features:", required_features)


Patient location: 28.6139 77.209
Required features: ['specialty:nephrology', 'equipment:dialysis_machine']


Service eligibility filtering

In [0]:
GRANT SELECT ON TABLE workspace.default.cleanfacility_credible
TO `chen.zhai.2008@gmail.com`;

GRANT SELECT ON TABLE workspace.default.cleanfacility_uncertain
TO `chen.zhai.2008@gmail.com`;

GRANT SELECT ON TABLE workspace.default.cleanfacility_unreliable
TO `chen.zhai.2008@gmail.com`;

In [0]:
GRANT SELECT ON TABLE workspace.default.cleanfacility_credible
TO `ajc3@andrew.cmu.edu`;

GRANT SELECT ON TABLE workspace.default.cleanfacility_uncertain
TO `ajc3@andrew.cmu.edu`;

GRANT SELECT ON TABLE workspace.default.cleanfacility_unreliable
TO `ajc3@andrew.cmu.edu`;

In [0]:
GRANT SELECT ON TABLE workspace.default.cleanfacility_credible
TO `minzhenn@andrew.cmu.edu`;

GRANT SELECT ON TABLE workspace.default.cleanfacility_uncertain
TO `minzhenn@andrew.cmu.edu`;

GRANT SELECT ON TABLE workspace.default.cleanfacility_unreliable
TO `minzhenn@andrew.cmu.edu`;

In [0]:

%python
from pyspark.sql import functions as F

# 1. Find facilities reporting the selected features
matched = (
    feature_entries
    .filter(F.col("feature").isin(required_features))
    .groupBy("matrix_tier", "facility_id")
    .agg(
        F.countDistinct("feature").alias("matched_count"),
        F.collect_set("feature").alias("matched_features"),
        F.flatten(F.collect_list("evidence")).alias("evidence")
    )
)

# 2. Keep facilities matching ALL requested features
filtered_facilities = (
    facility_matrix
    .join(matched, ["matrix_tier", "facility_id"], "inner")
    .filter(
        F.col("matched_count") == len(set(required_features))
    )
)

# 3. Display results
print("Required features:", required_features)
print("Facilities matching all requirements:", filtered_facilities.count())

display(
    filtered_facilities.select(
        "facility_name",
        "city",
        "reported_region",
        "matrix_tier",
        "matched_features",
        "evidence",
        "source_urls"
    )
)


Required features: ['specialty:nephrology', 'equipment:dialysis_machine']
Facilities matching all requirements: 58


facility_name city reported_region matrix_tier matched_features evidence source_urls Heritage Hospitals Varanasi Uttar Pradesh credible List(equipment:dialysis_machine, specialty:nephrology) List(Hemodialysis machine, nephrology) ["http://www.healthinsuranceindia.org/cashless_hospital_india/uttar_pradesh.html","https://heritagehospitals.com/","http://www.onefivenine.com/india/Listing/Town/hospitals/Gorakhpur/Barhalganj","https://heritagehospitals.com/departments/specialities/orthopedics-bone-diseases","https://heritagehospitals.com/doctor-detail/dr-ramesh-kumar-verma","https://heritagehospitals.com/departments/specialities/dentistry","https://www.apollohospitals.com/doctors/opthalmologist/chennai/dr-aparna-bhatnagar","https://heritagehospitals.com/departments/specialities/orthopedics-bone-diseases","https://heritagehospitals.com/departments/specialities/cardiology-heart","https://www.justdial.com/Raipur-Chhattisgarh/Dr-Kishore-Jha-Heritage-Hospital-Near-Timber-Bhawan-Devendra-Nagar-Sector-2-Raipur-Ganj/9999PX771-X771-130603151743-Z4Y9_BZDET","https://www.hospitalinformationindia.com/2014/10/hospitals-in-agra/","https://www.cardiologistindia.com/chhattisgarh/raipur","https://heritagehospitalgorakhpur.com/","https://www.justdial.com/Varanasi/Heritage-Hospitals-Ltd-Near-B-H-U-Main-Gate-Lanka/0542PX542-X542-000241147735-L5R6_BZDET","http://www.onefivenine.com/india/Listing/Locality/hospitals/Varanasi/Varanasi/Chowk","https://www.sg.jupiteryellowdetail.com/Agra/search-by-listings/hospital-/madia-katra-/kadambari-hospital/333926.jws","https://www.hexahealth.com/raipur/hospital/agrasen-hospital-bhavan","https://www.doctoriduniya.com/clinic/varanasi/omega-plus-varanasi","https://www.hexahealth.com/raipur/hospital/aayush-hospital-and-maternity-home","https://www.hexahealth.com/raipur/hospital/shri-venkatesh-superspeciality-hospital-sector-8-a","https://www.hexahealth.com/raipur/hospital/suyash-hospital-raipur","https://heritagehospitals.com/bloodbank/16","https://heritagehospitals.com/doctors","https://heritagehospitals.com/diagnostic","https://heritagehospitals.com/surgery","https://www.medifee.com/dc-in-varanasi","https://www.ketto.org/blog/cancer-hospitals-in-varanasi","https://heritagehospitals.com/departments/specialities/dentistry","https://heritagehospitals.com/departments/specialities/psychiatry","https://www.impactguru.com/info/cancer-hospitals-in-varanasi/","https://www.proptiger.com/varanasi/ramnagar-63125/hospitals","https://www.scribd.com/document/825053584/LKO-HOSPITAL","https://kashibanaras.com/hospital/","https://www.practo.com/varanasi/doctor/dr-amar-anupam-dentist","https://www.hexahealth.com/raipur/hospital/dr-renuka-sharma-eye-clinic-shankar-nagar","https://www.gangoshrihospital.com/top-5-hospital-in-varanasi/","http://www.heritagehospitals.com/","https://www.hexahealth.com/raipur/hospital/dks-super-speciality-hospital","https://varanasicityinindia.blogspot.com/2014/08/hospital-in-varanasi.html","https://www.justdial.com/Rampur/Heritage-Hospital-Near-Gandhi-Samadhi-Civil-Lines/9999P5952-5952-221011131526-P4U9_BZDET/photos","https://www.hexahealth.com/raipur/hospital/vasudev-multispeciality-hospital","https://www.mappls.com/9ibpyf","https://www.facebook.com/1016623541801270","https://www.facebook.com/100353332564085","https://www.justdial.com/Gorakhpur/Heritage-Hospital-Near-Chandi-Mata-Mandir-Harsewakpur-No-2-Khajanchi-Chowk/9999PX551-X551-240405233342-M8H1_BZDET","https://www.mappls.com/place-heritage+hospital-tirupati+vihar+colony-sukhravali-aligarh-uttar+pradesh-202002-z6a445@zdata=MjcuOTEzNDY2Kzc4LjExMjk5MSsxNyt6NmE0NDUrKw==ed","https://heritagehospitals.com/centerofexcellence/6","https://mapcarta.com/N7498563199","https://www.facebook.com/1753854644933091","https://www.sulekha.com/government-hospitals/varanasi"] Pankaj Multispeciality Hospital Sangli Maharashtra credible List(equipment:dialysis_machine, specialty:nephrology) List(Dialysis equipment, nephrology) ["https://www.vinsfertility.com/s/maharashtra/oppo

KNN

In [0]:

%python
# ============================================================
# CareProof - Step 2: Geographic KNN
#
# INPUT:
#   filtered_facilities (from Service Eligibility Filtering)
#   patient_lat
#   patient_lon
#   required_features
#
# OUTPUT:
#   knn_candidates (10 nearest matching facilities)
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import Window
import math

# ------------------------------------------------------------
# 1. SETTINGS
# ------------------------------------------------------------

K = 10
EARTH_RADIUS_KM = 6371.0
FAR_WARNING_KM = 100

# ------------------------------------------------------------
# 2. READ PREVIOUS STEP'S OUTPUT
# ------------------------------------------------------------

# IMPORTANT:
# We do NOT reload all facilities.
# We ONLY use facilities that passed the user's feature filter.

knn_pool = filtered_facilities

print("User location:", patient_lat, patient_lon)
print("Required features:", required_features)
print("Facilities passing feature filter:", knn_pool.count())

# ------------------------------------------------------------
# 3. VALIDATE USER COORDINATES
# ------------------------------------------------------------

patient_lat = float(patient_lat)
patient_lon = float(patient_lon)

if not (
    math.isfinite(patient_lat)
    and math.isfinite(patient_lon)
    and -90 <= patient_lat <= 90
    and -180 <= patient_lon <= 180
):
    raise ValueError("Invalid patient coordinates")

# ------------------------------------------------------------
# 4. CLEAN FACILITY COORDINATES
# ------------------------------------------------------------

knn_pool = (
    knn_pool
    .withColumn(
        "hospital_lat",
        F.expr("try_cast(latitude_clean AS DOUBLE)")
    )
    .withColumn(
        "hospital_lon",
        F.expr("try_cast(longitude_clean AS DOUBLE)")
    )
    .filter(
        F.col("hospital_lat").between(6, 37.5)
        & F.col("hospital_lon").between(68, 97.5)
    )
    .filter(
        F.col("matrix_tier").isin("credible", "uncertain")
    )
)

print("Candidates with plausible coordinates:", knn_pool.count())

# ------------------------------------------------------------
# 5. HAVERSINE DISTANCE
# ------------------------------------------------------------

lat1 = F.radians(F.lit(patient_lat))
lon1 = F.radians(F.lit(patient_lon))

lat2 = F.radians(F.col("hospital_lat"))
lon2 = F.radians(F.col("hospital_lon"))

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    F.pow(F.sin(dlat / 2), 2)
    + F.cos(lat1)
    * F.cos(lat2)
    * F.pow(F.sin(dlon / 2), 2)
)

# Clamp floating-point error to the valid asin domain.
a = F.least(F.lit(1.0), F.greatest(F.lit(0.0), a))

distance = (
    2 * EARTH_RADIUS_KM * F.asin(F.sqrt(a))
)

knn_pool = knn_pool.withColumn(
    "distance_km_exact",
    distance
)

# ------------------------------------------------------------
# 6. FIND K NEAREST MATCHING FACILITIES
# ------------------------------------------------------------

# Rank by unrounded distance, then break ties consistently.
# No other feature filtering happens here.

nearest = (
    knn_pool
    .orderBy(
        F.col("distance_km_exact").asc(),
        F.col("facility_id").asc()
    )
    .limit(K)
)

ranking_window = Window.orderBy(
    F.col("distance_km_exact").asc(),
    F.col("facility_id").asc()
)

knn_candidates = (
    nearest
    .withColumn(
        "knn_rank",
        F.row_number().over(ranking_window)
    )
    .withColumn(
        "distance_km",
        F.round("distance_km_exact", 2)
    )
    .withColumn(
        "far_warning",
        F.col("distance_km_exact") > FAR_WARNING_KM
    )
)

# ------------------------------------------------------------
# 7. SAVE FOR PROMETHEE II
# ------------------------------------------------------------

knn_candidates.createOrReplaceTempView(
    "knn_candidates"
)

# ------------------------------------------------------------
# 8. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n========== KNN RESULTS ==========")

print("Requested features:", required_features)
print("Patient coordinates:", patient_lat, patient_lon)
print("Number of nearest candidates:", knn_candidates.count())

display(
    knn_candidates.select(
        "knn_rank",
        "facility_name",
        "city",
        "reported_region",
        "matrix_tier",
        "distance_km",
        "far_warning",
        "matched_features",
        "evidence",
        "source_urls"
    ).orderBy("knn_rank")
)


User location: 28.6139 77.209
Required features: ['specialty:nephrology', 'equipment:dialysis_machine']
Facilities passing feature filter: 58
Candidates with plausible coordinates: 58


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(



========== KNN RESULTS ==========
Requested features: ['specialty:nephrology', 'equipment:dialysis_machine']
Patient coordinates: 28.6139 77.209
Number of nearest candidates: 10


knn_rank facility_name city reported_region matrix_tier distance_km far_warning matched_features evidence source_urls 1 Alshifa Multispeciality Hospital New Delhi Delhi credible 10.35 false List(equipment:dialysis_machine, specialty:nephrology) List(Dialysis machines, nephrology) ["http://www.onefivenine.com/india/villages/North-East-Delhi/North-East-Delhi/Jasola-Jasola","https://www.livemint.com/news/india/covid-vaccination-delhi-list-of-private-hospitals-administering-vaccines-11614740576681.html","https://alshifahospital.net/specialties/best-dental-specialists-in-jamia-nagar-okhla-new-delhi","https://alshifahospital.net/pediatric/dr-sumbul-warsi/","https://alshifahospital.net/services/physiotherapy-service-in-okhla-jamia-nagar-new-delhi/","https://alshifahospital.net/services/emergency-services/","https://alshifahospital.net/orthopaedics/dr-anand-m-vadehra/","https://alshifahospital.net/specialties/","https://chotu.com/local/dental-clinics-in-okhla-new-delhi-delhi-ncr-delhi-ncr-india/ct-18418653/","https://www.1mg.com/doctors/dr-anand-vadehra-orthopaedics-in-delhi/DR-7kl13","https://www.99acres.com/jasola-vihar-south-delhi-overview-piffid","https://alshifahospital.net/contact-us/","https://alshifahospital.net/services/","https://alshifahospital.net/specialties/best-dental-specialists-in-jamia-nagar-okhla-new-delhi/","http://www.cashlesshospitalindia.com/cashless-hospital-india/registered-private-nursing-homes-hospitals-delhi.html","https://www.diagnosticcentres.in/diagnostic-centres/delhi/new-delhi/sr-imaging-and-diagnostics.html","http://www.cashlesshospitalindia.com/registered-private-nursing-homes-hospitals-delhi.html",null] 2 Avantika Hospital Sahibabad Ghaziabad uncertain 16.28 false List(equipment:dialysis_machine, specialty:nephrology) List(Dialysis machines, nephrology) ["https://www.practo.com/ghaziabad/hospital/le-crest-medico-center-sahibabad","https://www.docindia.org/doctors/ghaziabad/dr-neetu-singh-gynecology","https://www.medicineindia.org/hospitals-in-state/uttar-pradesh","https://www.practo.com/ghaziabad/clinic/khaneja-nursing-home-new-gandhi-nagar","https://www.practo.com/ghaziabad/clinic/dr-nitesh-sharma-indirapuram","https://www.justdial.com/Bulandshahr/Avantika-Hospital-Near-By-Ip-Degree-College-Siyana-Adda-Choraha-Tanda/9999P5732-5732-171231114220-G2R3_BZDET","https://www.practo.com/ghaziabad/clinic/siddhi-vinayak-maternity-and-family-clinic-pratap-vihar","https://www.practo.com/ghaziabad/doctor/dr-neha-poddar-gynecologist-obstetrician-1","https://www.practo.com/ghaziabad/clinic/varenya-clinic-vasundhara","https://www.practo.com/ghaziabad/clinic/tooth-tamers-govindpuram/dentist-ps-8","https://www.practo.com/ghaziabad/hospital/swami-nand-lal-jee-hospital-pvt-ltd-sahibabad","https://www.practo.com/ghaziabad/clinic/avantika-hospital-indirapuram-1","https://www.practo.com/ghaziabad/clinic/dr-devesh-jain-dental-clinic-vaishali","https://www.practo.com/ghaziabad/clinic/dr-jha-s-digital-dentistrry-shakti-khand-i","https://www.practo.com/ghaziabad/hospital/paras-hospital-12-vaishali-1","http://www.dieticiankirti.com/service.php","https://www.medindia.net/directories/labs/unique-pathology-laboratory-ghaziabad-uttar-pradesh-41465-1.htm","https://www.sulekha.com/sanjivani-physiotherapy-fitness-studio-indirapuram-ghaziabad-contact-address","https://www.practo.com/ghaziabad/doctor/dr-amit-kumar-11-general-physician-11","https://www.justdial.com/Ghaziabad/Avantika-Hospital-Near-Swarn-Jayanti-Park-Niti-Khand-2-Indirapuram/011PXX11-XX11-130826134713-R6X9_BZDET","https://www.facebook.com/106273949180140","https://www.practo.com/ghaziabad/doctor/dr-vidya-sinha-gynecologist-obstetrician","https://www.practo.com/ghaziabad/doctor/shruti-khurana-general-physician","https://www.practo.com/ghaziabad/doctor/dr-neha-saxena-gynecologist-obstetrician","https://www.facebook.com/103712331324445","https://www.practo.com/ghaziabad/clinic/anaya-maternity-clinic-indirapuram","https://www.justdial.com/Ghaziabad/General-Physician-Doctors

AHP

In [0]:
%python
from itertools import combinations
from math import exp, log

criteria = ("distance", "facilities", "personnel", "reliability")

# Three adjusted responses, in the questionnaire's six-row order.
# Positive favors the left side, negative favors the right, and 1 means equal.
responses = (
    (-5,  3, -3, 7, 5, -3),
    (-5, -5, -3, 3, 5,  3),
    (-5,  3,  1, 7, 7, -3),
)

n = len(criteria)
matrix = [[1.0] * n for _ in range(n)]

# Combine each comparison using the geometric mean.
for (left, right), ratings in zip(combinations(range(n), 2), zip(*responses)):
    mean_log = sum(
        log(rating) if rating > 0 else -log(-rating)
        for rating in ratings
    ) / len(ratings)
    matrix[left][right] = exp(mean_log)
    matrix[right][left] = 1 / matrix[left][right]

# Calculate the normalized AHP weights.
weights = [1 / n] * n
for _ in range(1000):
    product = [
        sum(value * weight for value, weight in zip(row, weights))
        for row in matrix
    ]
    eigenvalue = sum(product)
    next_weights = [value / eigenvalue for value in product]
    if max(abs(a - b) for a, b in zip(weights, next_weights)) < 1e-12:
        weights = next_weights
        break
    weights = next_weights
else:
    raise RuntimeError("AHP weights did not converge.")

DEFAULT_WEIGHTS = dict(zip(criteria, weights))
GROUP_CONSISTENCY_RATIO = (eigenvalue - n) / ((n - 1) * 0.90)

for section, weight in DEFAULT_WEIGHTS.items():
    print(f"{section}: {weight:.2%}")
print(f"Group CR: {GROUP_CONSISTENCY_RATIO:.4f}")

distance: 10.63%
facilities: 63.40%
personnel: 10.28%
reliability: 15.69%
Group CR: 0.0288


In [0]:
%python
facility_features = (
    "reported_capacity",
    "emergency_24_7",
    "dental_emergency_24_7",
    "mri_equipment",
    "mri_scan",
    "ct_scanner",
    "ct_scan",
    "blood_draw",
    "dialysis_machine",
    "hemodialysis",
    "peritoneal_dialysis",
    "ventilator",
    "ultrasound_equipment",
    "ultrasound_scan",
    "xray_equipment",
    "nicu",
    "icu",
    "cesarean_delivery",
)

FACILITY_DEFAULT_WEIGHTS = {
    feature: DEFAULT_WEIGHTS["facilities"] / len(facility_features)
    for feature in facility_features
}

print(f"Facilities total: {sum(FACILITY_DEFAULT_WEIGHTS.values()) * 100:.4f} points")
print(f"Each feature: {FACILITY_DEFAULT_WEIGHTS['mri_scan'] * 100:.4f} points")

Facilities total: 63.4047 points
Each feature: 3.5225 points


User weight adjustment + Promethee II

In [0]:

%python
# ============================================================
# CAREPROOF — PROMETHEE II
# CELL 1: CONFIGURATION AND USER CONTROLS
# ============================================================

import math
import re
from collections import defaultdict
from pyspark.sql import functions as F

FACILITY_KEYS = [
    "reported_capacity",
    "emergency_24_7",
    "dental_emergency_24_7",
    "mri_equipment",
    "mri_scan",
    "ct_scanner",
    "ct_scan",
    "blood_draw",
    "dialysis_machine",
    "hemodialysis",
    "peritoneal_dialysis",
    "ventilator",
    "ultrasound_equipment",
    "ultrasound_scan",
    "xray_equipment",
    "nicu",
    "icu",
    "cesarean_delivery",
]

# These are defaults for the prototype.
# Replace with AHP-derived values when available.
DEFAULT_WEIGHTS = {
    "personnel": 20.0,
    "facilities": 45.0,
    "distance": 20.0,
    "credibility": 15.0,
}

DEFAULT_A = 70.0
DEFAULT_B = 30.0
PRIORITY_BOOST = 2.0

# Provisional preference thresholds
DISTANCE_Q = 1.0
DISTANCE_P = 20.0
COUNT_Q = 0.0
COUNT_P = math.log(2.0)

KEYS = ["matrix_tier", "facility_id"]


def ensure_text(name, default, label):
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, str(default), label)


# Four main-category weights
ensure_text(
    "w_personnel", 20, "1. Personnel (%)"
)
ensure_text(
    "w_facilities", 45, "2. Facilities (%)"
)
ensure_text(
    "w_distance", 20, "3. Distance (%)"
)
ensure_text(
    "w_credibility", 15, "4. Credibility (%)"
)

# A/B split INSIDE Facilities
ensure_text(
    "w_required_A", 70,
    "5. Required Facilities A (%)"
)
ensure_text(
    "w_preferred_B", 30,
    "6. Additional Preferences B (%)"
)

# The earlier AI-input stage must provide
# ai_required_facilities (exactly three feature keys).
try:
    dbutils.widgets.get("ai_required_facilities")
except Exception:
    dbutils.widgets.multiselect(
        "ai_required_facilities",
        "mri_scan",
        FACILITY_KEYS,
        "AI-Suggested Facilities (choose 3)"
    )

print("PROMETHEE II configuration ready.")
print("Set the four category weights and A/B weights.")
print("Select exactly three required facilities.")


PROMETHEE II configuration ready.
Set the four category weights and A/B weights.
Select exactly three required facilities.


In [0]:

%python
# ============================================================
# CELL 2: REMOVE OLD PREFERENCES DROPDOWN
# ============================================================

try:
    dbutils.widgets.remove("preferred_facilities")
except Exception:
    pass

print("Preference menu cleared for refresh.")


Preference menu cleared for refresh.


In [0]:

%python
# ============================================================
# CELL 3: DYNAMIC FACILITIES MULTISELECT
# ============================================================

required_list = [
    x.strip()
    for x in dbutils.widgets.get(
        "ai_required_facilities"
    ).split(",")
    if x.strip()
]

required_list = list(dict.fromkeys(required_list))

if len(required_list) != 3:
    raise ValueError(
        "Exactly 3 AI-suggested facilities must be selected."
    )

if not set(required_list).issubset(set(FACILITY_KEYS)):
    raise ValueError(
        "One or more required facilities are not recognized."
    )

# Capacity is not itself a medical treatment/service.
if "reported_capacity" in required_list:
    raise ValueError(
        "reported_capacity cannot be used as a required "
        "medical service. It is a ranking preference."
    )

required_facilities = set(required_list)

optional_facilities = [
    key for key in FACILITY_KEYS
    if key not in required_facilities
]

dbutils.widgets.multiselect(
    "preferred_facilities",
    "none",
    ["none"] + optional_facilities,
    "7. Additional Facilities Preferences"
)

print("AI-required facilities:")
for key in required_list:
    print("  REQUIRED:", key)

print("\nOptional preference menu:")
for key in optional_facilities:
    print("  OPTIONAL:", key)

print("\nSelect any additional preferences in the menu.")


AI-required facilities:
  REQUIRED: mri_scan
  REQUIRED: ct_scanner
  REQUIRED: ct_scan

Optional preference menu:
  OPTIONAL: reported_capacity
  OPTIONAL: emergency_24_7
  OPTIONAL: dental_emergency_24_7
  OPTIONAL: mri_equipment
  OPTIONAL: blood_draw
  OPTIONAL: dialysis_machine
  OPTIONAL: hemodialysis
  OPTIONAL: peritoneal_dialysis
  OPTIONAL: ventilator
  OPTIONAL: ultrasound_equipment
  OPTIONAL: ultrasound_scan
  OPTIONAL: xray_equipment
  OPTIONAL: nicu
  OPTIONAL: icu
  OPTIONAL: cesarean_delivery

Select any additional preferences in the menu.


In [0]:

%python
# ============================================================
# CAREPROOF — PROMETHEE II
# CELL 4: COMPLETE RANKING ENGINE
#
# REQUIRED DATAFRAMES FROM PREVIOUS PIPELINE:
#   knn_candidates
#   feature_summary
#
# OUTPUTS:
#   default_promethee_results
#   promethee_results
#   promethee_explanations
#   promethee_weights
#   promethee_feature_coverage
#
# No independent KNN or service filtering here.
# ============================================================

import math
from collections import defaultdict
from pyspark.sql import functions as F

# ============================================================
# 1. VALIDATE INPUT DATA
# ============================================================

if "knn_candidates" not in globals():
    raise ValueError(
        "Run the updated eligibility filtering and KNN first."
    )

if "feature_summary" not in globals():
    raise ValueError(
        "Missing feature_summary. Run the updated feature "
        "extraction and eligibility filtering first."
    )

required_knn_columns = {
    "matrix_tier",
    "facility_id",
    "facility_name",
    "distance_km",
    "staffing_vector",
    "facilities_vector",
    "matched_required",
}

missing_columns = (
    required_knn_columns - set(knn_candidates.columns)
)

if missing_columns:
    raise ValueError(
        f"KNN is missing columns: {sorted(missing_columns)}"
    )

# ============================================================
# 2. READ CURRENT AI REQUIREMENTS AND PREFERENCES
# ============================================================

required_list = list(dict.fromkeys(
    x.strip()
    for x in dbutils.widgets.get(
        "ai_required_facilities"
    ).split(",")
    if x.strip()
))

if len(required_list) != 3:
    raise ValueError(
        "Exactly 3 required facilities must be selected."
    )

if not set(required_list).issubset(set(FACILITY_KEYS)):
    raise ValueError("Unknown required facility.")

if "reported_capacity" in required_list:
    raise ValueError(
        "Capacity cannot be a required medical service."
    )

required_set = set(required_list)

preferred_list = list(dict.fromkeys(
    x.strip()
    for x in dbutils.widgets.get(
        "preferred_facilities"
    ).split(",")
    if x.strip() and x.strip() != "none"
))

preferred_set = set(preferred_list)

if preferred_set & required_set:
    raise ValueError(
        "A required facility was also selected as preferred. "
        "Refresh the preferences dropdown."
    )

optional_list = [
    key for key in FACILITY_KEYS
    if key not in required_set
]

if not preferred_set.issubset(set(optional_list)):
    raise ValueError("Unknown preferred facility.")

# ============================================================
# 3. NORMALIZE THE FOUR CATEGORY WEIGHTS
# ============================================================

def read_weight(name):
    value = float(dbutils.widgets.get(name))

    if not math.isfinite(value) or value < 0:
        raise ValueError(
            f"{name} must be a nonnegative finite number."
        )

    return value


raw_parent = {
    "personnel": read_weight("w_personnel"),
    "facilities": read_weight("w_facilities"),
    "distance": read_weight("w_distance"),
    "credibility": read_weight("w_credibility"),
}

parent_total = sum(raw_parent.values())

if parent_total <= 0:
    raise ValueError(
        "At least one category weight must be positive."
    )

custom_parent = {
    key: value / parent_total
    for key, value in raw_parent.items()
}

default_total = sum(DEFAULT_WEIGHTS.values())

default_parent = {
    key: value / default_total
    for key, value in DEFAULT_WEIGHTS.items()
}

# ============================================================
# 4. NORMALIZE THE A/B SPLIT
# ============================================================

raw_A = read_weight("w_required_A")
raw_B = read_weight("w_preferred_B")

if raw_A + raw_B <= 0:
    raise ValueError(
        "Required A and Preferred B cannot both be zero."
    )

custom_A = raw_A / (raw_A + raw_B)
custom_B = raw_B / (raw_A + raw_B)

default_A = DEFAULT_A / (DEFAULT_A + DEFAULT_B)
default_B = DEFAULT_B / (DEFAULT_A + DEFAULT_B)

# ============================================================
# 5. LOAD EXACTLY THE EXISTING KNN SHORTLIST
# ============================================================

candidate_rows = (
    knn_candidates
    .orderBy("distance_km", "facility_id")
    .collect()
)

if not candidate_rows:
    raise ValueError("KNN returned zero candidates.")

candidate_keys = {
    (row["matrix_tier"], str(row["facility_id"]))
    for row in candidate_rows
}

# ============================================================
# 6. LOAD FACILITY EVIDENCE FOR THE SAME CANDIDATES
# ============================================================

candidate_ids = (
    knn_candidates
    .select(*KEYS)
    .distinct()
)

evidence_rows = (
    feature_summary
    .join(candidate_ids, KEYS, "inner")
    .select(
        *KEYS,
        "medical_feature",
        "has_positive_mention",
        "has_negative_mention",
        "evidence_quotes",
    )
    .collect()
)

evidence_map = defaultdict(dict)
evidence_quotes = defaultdict(dict)
evidence_status = defaultdict(dict)

for row in evidence_rows:
    key = (
        row["matrix_tier"],
        str(row["facility_id"])
    )

    feature = row["medical_feature"]

    positive = row["has_positive_mention"] == 1
    negative = row["has_negative_mention"] == 1

    if negative and positive:
        status = "conflicting_evidence"
        value = None
    elif negative:
        status = "negative_claim_review"
        value = None
    elif positive:
        status = "positive_mention"
        value = 1.0
    else:
        status = "not_documented"
        value = 0.0

    evidence_map[key][feature] = value
    evidence_status[key][feature] = status

    evidence_quotes[key][feature] = (
        row["evidence_quotes"] or []
    )

# ============================================================
# 7. PREPARE A VALUE VECTOR FOR EACH HOSPITAL
# ============================================================

def as_dict(value):
    if isinstance(value, dict):
        return value
    return value.asDict(recursive=True)


def numeric_vector_value(vector, name):
    for item in vector or []:
        entry = as_dict(item)

        if entry.get("feature") != name:
            continue

        try:
            value = float(entry["value"])
        except (TypeError, ValueError):
            return None

        if math.isfinite(value) and value >= 0:
            return value

    return None


hospitals = []

for row in candidate_rows:
    r = row.asDict(recursive=True)

    key = (
        r["matrix_tier"],
        str(r["facility_id"])
    )

    distance = r.get("distance_km_exact")

    if distance is None:
        distance = r["distance_km"]

    distance = float(distance)

    if not math.isfinite(distance) or distance < 0:
        raise ValueError(
            f"Invalid distance for {r['facility_id']}"
        )

    doctor_count = numeric_vector_value(
        r.get("staffing_vector"),
        "doctor_count"
    )

    capacity = numeric_vector_value(
        r.get("facilities_vector"),
        "reported_capacity"
    )

    # Prototype outlier quarantine.
    # These limits are NOT medical standards.
    if doctor_count is not None and doctor_count > 1000:
        doctor_count = None

    if capacity is not None and capacity > 3000:
        capacity = None

    values = {
        "personnel": doctor_count,
        "distance": distance,
        "credibility": (
            1.0 if r["matrix_tier"] == "credible"
            else 0.0 if r["matrix_tier"] == "uncertain"
            else None
        ),
    }

    status = {}
    quotes = {}

    for feature in FACILITY_KEYS:

        if feature == "reported_capacity":
            values[feature] = capacity
            status[feature] = (
                "reported_numeric_value"
                if capacity is not None
                else "unknown"
            )
            quotes[feature] = []
            continue

        values[feature] = evidence_map[key].get(
            feature, 0.0
        )

        status[feature] = evidence_status[key].get(
            feature, "not_documented"
        )

        quotes[feature] = evidence_quotes[key].get(
            feature, []
        )

    # Recheck that these facilities satisfy the CURRENT
    # set of three requirements.
    matched_now = [
        feature for feature in required_list
        if values[feature] == 1.0
    ]

    # Never silently rank a stale KNN shortlist created
    # using different AI requirements.
    if not matched_now:
        raise ValueError(
            f"{r['facility_name']} does not match any "
            "CURRENT required facility. Rerun the OR "
            "eligibility filter and KNN."
        )

    original_matched = set(r.get("matched_required") or [])

    if original_matched != set(matched_now):
        raise ValueError(
            f"Stale KNN requirement data for "
            f"{r['facility_name']}. Rerun eligibility "
            "filtering and KNN with the current three "
            "requirements."
        )

    hospitals.append({
        "facility_id": str(r["facility_id"]),
        "facility_name": r["facility_name"],
        "matrix_tier": r["matrix_tier"],
        "knn_rank": int(r.get("knn_rank") or 0),
        "distance_km": distance,
        "matched_required": matched_now,
        "matched_count": len(matched_now),
        "values": values,
        "evidence_status": status,
        "evidence_quotes": quotes,
        "source_urls": r.get("source_urls"),
    })

print("KNN candidates loaded:", len(hospitals))
print("Required facilities:", required_list)

# ============================================================
# 8. CREATE ONE SHARED FACILITIES WEIGHT VECTOR
# ============================================================

def calculate_facility_weights(
    parent,
    a_share,
    b_share,
    selected,
):
    """
    A: equal weight over the 3 required facilities.

    B: equal baseline over all 15 optional facilities;
       user-selected optional facilities receive 2x.

    Weights are search-wide, NOT hospital-specific.
    """

    facilities_parent = parent["facilities"]

    required_weights = {
        key: (
            facilities_parent
            * a_share
            / len(required_list)
        )
        for key in required_list
    }

    optional_importance = {
        key: (
            PRIORITY_BOOST
            if key in selected
            else 1.0
        )
        for key in optional_list
    }

    optional_total = sum(
        optional_importance.values()
    )

    optional_weights = {
        key: (
            facilities_parent
            * b_share
            * optional_importance[key]
            / optional_total
        )
        for key in optional_list
    }

    combined = {
        **required_weights,
        **optional_weights,
    }

    assert set(combined) == set(FACILITY_KEYS)

    assert abs(
        sum(combined.values()) - facilities_parent
    ) < 1e-9

    return combined


default_facility_weights = calculate_facility_weights(
    default_parent,
    default_A,
    default_B,
    selected=set()
)

custom_facility_weights = calculate_facility_weights(
    custom_parent,
    custom_A,
    custom_B,
    selected=preferred_set
)

# ============================================================
# 9. BUILD PROMETHEE II CRITERIA
# ============================================================

def build_criteria(parent, facility_weights):

    criteria = [
        {
            "name": "personnel",
            "group": "personnel",
            "weight": parent["personnel"],
            "direction": "max",
            "transform": "log",
            "q": COUNT_Q,
            "p": COUNT_P,
        },
        {
            "name": "distance",
            "group": "distance",
            "weight": parent["distance"],
            "direction": "min",
            "transform": "raw",
            "q": DISTANCE_Q,
            "p": DISTANCE_P,
        },
        {
            "name": "credibility",
            "group": "credibility",
            "weight": parent["credibility"],
            "direction": "max",
            "transform": "raw",
            "q": 0.0,
            "p": 1.0,
        },
    ]

    for feature in FACILITY_KEYS:

        is_capacity = (
            feature == "reported_capacity"
        )

        criteria.append({
            "name": feature,
            "group": (
                "required_A"
                if feature in required_set
                else "preferred_B"
            ),
            "weight": facility_weights[feature],
            "direction": "max",
            "transform": (
                "log" if is_capacity else "raw"
            ),
            "q": 0.0,
            "p": (
                COUNT_P if is_capacity else 1.0
            ),
        })

    if len(criteria) != 21:
        raise ValueError(
            "Expected 21 criteria: "
            "Personnel + Distance + Credibility + 18 Facilities."
        )

    if abs(
        sum(c["weight"] for c in criteria) - 1.0
    ) > 1e-9:
        raise ValueError("Criterion weights do not sum to 1.")

    return criteria


default_criteria = build_criteria(
    default_parent,
    default_facility_weights
)

custom_criteria = build_criteria(
    custom_parent,
    custom_facility_weights
)

# ============================================================
# 10. PROMETHEE II PREFERENCE FUNCTION
# ============================================================

def preference(a, b, criterion):

    name = criterion["name"]

    va = a["values"].get(name)
    vb = b["values"].get(name)

    # Missing/uncertain value:
    # no directed preference in either direction.
    #
    # Crucially, no per-hospital weight normalization.
    if va is None or vb is None:
        return 0.0

    va = float(va)
    vb = float(vb)

    if criterion["transform"] == "log":
        va = math.log1p(va)
        vb = math.log1p(vb)

    if criterion["direction"] == "min":
        advantage = vb - va
    else:
        advantage = va - vb

    q = criterion["q"]
    p = criterion["p"]

    if advantage <= q:
        return 0.0

    if advantage >= p:
        return 1.0

    return (advantage - q) / (p - q)


# ============================================================
# 11. RUN PROMETHEE II
# ============================================================

def run_promethee(hospitals, criteria):

    n = len(hospitals)

    outgoing = [0.0] * n
    incoming = [0.0] * n

    contributions = [
        defaultdict(float)
        for _ in range(n)
    ]

    # Compare every unordered pair once.
    for i in range(n):

        for j in range(i + 1, n):

            h_i = hospitals[i]
            h_j = hospitals[j]

            pref_ij = 0.0
            pref_ji = 0.0

            for criterion in criteria:

                weight = criterion["weight"]

                if weight <= 0:
                    continue

                p_ij = (
                    weight
                    * preference(h_i, h_j, criterion)
                )

                p_ji = (
                    weight
                    * preference(h_j, h_i, criterion)
                )

                pref_ij += p_ij
                pref_ji += p_ji

                contributions[i][
                    criterion["name"]
                ] += p_ij - p_ji

                contributions[j][
                    criterion["name"]
                ] += p_ji - p_ij

            outgoing[i] += pref_ij
            incoming[i] += pref_ji

            outgoing[j] += pref_ji
            incoming[j] += pref_ij

    denominator = max(n - 1, 1)

    results = []

    for i, hospital in enumerate(hospitals):

        positive_flow = (
            outgoing[i] / denominator
        )

        negative_flow = (
            incoming[i] / denominator
        )

        net_flow = (
            positive_flow - negative_flow
        )

        criterion_contributions = {
            key: value / denominator
            for key, value
            in contributions[i].items()
        }

        results.append({
            **hospital,
            "positive_flow": positive_flow,
            "negative_flow": negative_flow,
            "net_flow": net_flow,
            "criterion_contributions":
                criterion_contributions,
        })

    return sorted(
        results,
        key=lambda h: (
            -h["net_flow"],
            h["distance_km"],
            h["facility_id"]
        )
    )


# ============================================================
# 12. GENERATE DEFAULT AND CUSTOM RANKINGS
# ============================================================

default_ranked = run_promethee(
    hospitals,
    default_criteria
)

custom_ranked = run_promethee(
    hospitals,
    custom_criteria
)

# ============================================================
# 13. CONVERT RESULTS TO SPARK DATAFRAMES
# ============================================================

RESULT_SCHEMA = """
    promethee_rank INT,
    knn_rank INT,
    facility_id STRING,
    facility_name STRING,
    matrix_tier STRING,
    distance_km DOUBLE,
    required_matched_count INT,
    matched_required STRING,
    missing_required STRING,
    positive_flow DOUBLE,
    negative_flow DOUBLE,
    net_flow DOUBLE
"""


def results_to_df(ranked):

    rows = []

    for rank, hospital in enumerate(ranked, 1):

        missing_required = [
            feature
            for feature in required_list
            if feature not in hospital["matched_required"]
        ]

        rows.append((
            int(rank),
            int(hospital["knn_rank"]),
            hospital["facility_id"],
            hospital["facility_name"],
            hospital["matrix_tier"],
            float(hospital["distance_km"]),
            int(hospital["matched_count"]),
            ", ".join(hospital["matched_required"]),
            ", ".join(missing_required),
            float(hospital["positive_flow"]),
            float(hospital["negative_flow"]),
            float(hospital["net_flow"]),
        ))

    return spark.createDataFrame(
        rows,
        schema=RESULT_SCHEMA
    )


default_promethee_results = results_to_df(
    default_ranked
)

promethee_results = results_to_df(
    custom_ranked
)

# ============================================================
# 14. CREATE EXPLANATIONS FOR THE CUSTOM RANKING
# ============================================================

EXPLANATION_SCHEMA = """
    promethee_rank INT,
    facility_id STRING,
    facility_name STRING,
    criterion STRING,
    group_name STRING,
    global_weight DOUBLE,
    recorded_value DOUBLE,
    evidence_status STRING,
    net_contribution DOUBLE,
    source_evidence STRING
"""

explanation_rows = []

for rank, hospital in enumerate(custom_ranked, 1):

    for criterion in custom_criteria:

        feature = criterion["name"]

        value = hospital["values"].get(feature)

        if feature in FACILITY_KEYS:

            status = hospital[
                "evidence_status"
            ].get(feature, "unknown")

            quotes = hospital[
                "evidence_quotes"
            ].get(feature, [])

        elif feature == "credibility":

            status = "record_quality_tier"
            quotes = []

        elif value is None:

            status = "unknown"
            quotes = []

        else:

            status = "reported_numeric_value"
            quotes = []

        explanation_rows.append((
            int(rank),
            hospital["facility_id"],
            hospital["facility_name"],
            feature,
            criterion["group"],
            float(criterion["weight"]),
            (
                float(value)
                if value is not None
                else None
            ),
            status,
            float(
                hospital[
                    "criterion_contributions"
                ].get(feature, 0.0)
            ),
            " | ".join(quotes),
        ))


promethee_explanations = spark.createDataFrame(
    explanation_rows,
    schema=EXPLANATION_SCHEMA
)

# ============================================================
# 15. CREATE SHARED WEIGHT CONFIGURATION TABLE
# ============================================================

WEIGHT_SCHEMA = """
    group_name STRING,
    criterion STRING,
    default_global_weight DOUBLE,
    customized_global_weight DOUBLE,
    is_required BOOLEAN,
    is_user_prioritized BOOLEAN
"""

default_weight_lookup = {
    c["name"]: c["weight"]
    for c in default_criteria
}

custom_weight_lookup = {
    c["name"]: c["weight"]
    for c in custom_criteria
}

weight_rows = []

for criterion in custom_criteria:

    name = criterion["name"]

    weight_rows.append((
        criterion["group"],
        name,
        float(default_weight_lookup[name]),
        float(custom_weight_lookup[name]),
        bool(name in required_set),
        bool(name in preferred_set),
    ))

promethee_weights = spark.createDataFrame(
    weight_rows,
    schema=WEIGHT_SCHEMA
)

# ============================================================
# 16. CREATE FACILITY COVERAGE REPORT
# ============================================================

COVERAGE_SCHEMA = """
    feature STRING,
    is_required BOOLEAN,
    is_user_prioritized BOOLEAN,
    positive_mention_count INT,
    unknown_or_conflict_count INT,
    total_candidates INT,
    customized_weight_pct DOUBLE
"""

coverage_rows = []

for feature in FACILITY_KEYS:

    positive = 0
    unknown = 0

    for hospital in hospitals:

        value = hospital["values"].get(feature)

        if feature == "reported_capacity":

            if value is not None:
                positive += 1
            else:
                unknown += 1

        elif value == 1.0:
            positive += 1

        elif value is None:
            unknown += 1

        # A zero here means no positive mention.
        # It is not confirmed service absence.

    coverage_rows.append((
        feature,
        bool(feature in required_set),
        bool(feature in preferred_set),
        int(positive),
        int(unknown),
        int(len(hospitals)),
        float(
            custom_facility_weights[feature] * 100
        ),
    ))

promethee_feature_coverage = spark.createDataFrame(
    coverage_rows,
    schema=COVERAGE_SCHEMA
)

# ============================================================
# 17. SAVE TEMP VIEWS FOR DASHBOARD / GENIE
# ============================================================

default_promethee_results.createOrReplaceTempView(
    "default_promethee_results"
)

promethee_results.createOrReplaceTempView(
    "promethee_results"
)

promethee_explanations.createOrReplaceTempView(
    "promethee_explanations"
)

promethee_weights.createOrReplaceTempView(
    "promethee_weights"
)

promethee_feature_coverage.createOrReplaceTempView(
    "promethee_feature_coverage"
)

# ============================================================
# 18. DISPLAY
# ============================================================

print("=" * 55)
print("CAREPROOF — PROMETHEE II COMPLETE")
print("=" * 55)

print("\nAI-required facilities:")
print(required_list)

print("\nAdditional user preferences:")
print(sorted(preferred_set))

print("\nDefault A/B split:")
print(f"Required A: {default_A:.1%}")
print(f"Preferred B: {default_B:.1%}")

print("\nCustomized A/B split:")
print(f"Required A: {custom_A:.1%}")
print(f"Preferred B: {custom_B:.1%}")

print("\nMain category weights:")
for category, weight in custom_parent.items():
    print(f"{category}: {weight:.1%}")

print("\n--- DEFAULT RANKING ---")
display(
    default_promethee_results
    .orderBy("promethee_rank")
)

print("\n--- USER-CUSTOMIZED RANKING ---")
display(
    promethee_results
    .orderBy("promethee_rank")
)

print("\n--- DEFAULT VS CUSTOM WEIGHTS ---")
display(
    promethee_weights
    .orderBy("group_name", "criterion")
)

print("\n--- FACILITIES EVIDENCE COVERAGE ---")
display(
    promethee_feature_coverage
    .orderBy("feature")
)

print("\n--- RANKING EXPLANATIONS ---")
display(
    promethee_explanations
    .orderBy("promethee_rank", "criterion")
)


In [0]:

%python
# ============================================================
# CAREPROOF — PROMETHEE II
# CELL 5: DEFAULT VS CUSTOM RANKING
# ============================================================

default_view = (
    default_promethee_results
    .select(
        "facility_id",
        "facility_name",
        F.col("promethee_rank").alias("default_rank"),
        F.col("net_flow").alias("default_net_flow")
    )
)

custom_view = (
    promethee_results
    .select(
        "facility_id",
        F.col("promethee_rank").alias("custom_rank"),
        F.col("net_flow").alias("custom_net_flow")
    )
)

ranking_comparison = (
    default_view
    .join(custom_view, "facility_id")
    .withColumn(
        "rank_improvement",
        F.col("default_rank") - F.col("custom_rank")
    )
    .orderBy("custom_rank")
)

ranking_comparison.createOrReplaceTempView(
    "ranking_comparison"
)

display(ranking_comparison)


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7346736786498663>, line 8
      2 # ============================================================
      3 # CAREPROOF — PROMETHEE II
      4 # CELL 5: DEFAULT VS CUSTOM RANKING
      5 # ============================================================
      7 default_view = (
----> 8     default_promethee_results
      9     .select(
     10         "facility_id",
     11         "facility_name",
     12         F.col("promethee_rank").alias("default_rank"),
     13         F.col("net_flow").alias("default_net_flow")
     14     )
     15 )
     17 custom_view = (
     18     promethee_results
     19     .select(
   (...)     23     )
     24 )
     26 ranking_comparison = (
     27     default_view
     28     .join(custom_view, "facility_id")
   (...)     33     .orderBy("custom_rank")
     34 )

NameError: name 'default_pro